# 🍔 Principal Component Analysis (PCA) Plots
## A Complete Tutorial Using McDonald's Menu Nutrition Data

---

### Course: TTK4260 - Multivariate Data Analysis

This notebook provides a **comprehensive tutorial** on **PCA visualization and plot interpretation**, aligned with the course slides on plots. Each section explains what each plot shows and how to extract meaningful insights.

---

### 📋 Plot Categories (Aligned with PCAplots.tex)

This notebook covers plots organized into **7 categories**:

| # | Category | Plots Covered |
|---|----------|---------------|
| **1** | **Model Interpretation** | Score Plot, Loading Plot, Biplot, Correlation Loading Plot |
| **2** | **Model Validation & Selection** | Scree Plot, Explained Variance, Cross-Validation, Eigenvalue Methods |
| **3** | **Outlier Detection & Diagnostics** | Hotelling's T², Q/SPE Statistic, T² vs Q Plot, Control Charts |
| **4** | **Fault Diagnosis** | T² Contribution, Q Contribution, Score Contribution |
| **5** | **Variable Analysis** | Loading Bar Chart, Loading Heatmap, VIP Plot, Communalities |
| **6** | **Residual Analysis** | Residual Heatmap, Predicted vs Actual, Q-Q Plot |
| **7** | **Process Monitoring (MSPC)** | Multivariate Control Charts |

---

### Dataset: McDonald's India Menu
- **141 menu items** across 7 categories
- **10 nutritional variables**: Energy, Protein, Fats, Carbohydrates, etc.
- Real-world data perfect for demonstrating multivariate analysis

### 📦 Dependency Installation

Run this cell to automatically install all required packages if they're not already available. This cell will check for each package and only install what's missing.

In [1]:
import subprocess
import sys

def install_if_missing(package_name, import_name=None):
    """
    Install a package if it's not already available.
    
    Args:
        package_name: Name of package to install via pip
        import_name: Name to use for import check (if different from package_name)
    """
    if import_name is None:
        import_name = package_name
    
    try:
        __import__(import_name)
        print(f"✅ {package_name} is already installed")
    except ImportError:
        print(f"📦 Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, "-q"])
        print(f"✅ {package_name} installed successfully")

# List of required packages
packages = [
    ('numpy', 'numpy'),
    ('pandas', 'pandas'),
    ('scipy', 'scipy'),
    ('scikit-learn', 'sklearn'),
    ('plotly', 'plotly'),
    ('ipywidgets', 'ipywidgets'),
]

print("="*70)
print("🔍 Checking and installing required dependencies...")
print("="*70)

for package_name, import_name in packages:
    install_if_missing(package_name, import_name)

print("\n" + "="*70)
print("✅ All dependencies are ready!")
print("="*70)

🔍 Checking and installing required dependencies...
✅ numpy is already installed
✅ pandas is already installed
✅ scipy is already installed
✅ scikit-learn is already installed
✅ plotly is already installed
✅ ipywidgets is already installed

✅ All dependencies are ready!


---
# Part 1: Setup and Data Loading
---

### 1.1 Import Libraries

We use:
- **numpy/pandas**: Data manipulation
- **plotly**: Interactive visualizations
- **sklearn**: PCA implementation and preprocessing
- **scipy**: Statistical functions

In [2]:
# Core libraries
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import f as f_dist, chi2
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import KFold

# Interactive plotting with Plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Color scheme - McDonald's inspired
COLORS = {
    'mcd_red': '#DA291C',
    'mcd_yellow': '#FFC72C',
    'blue': '#1f77b4',
    'green': '#2ca02c',
    'orange': '#ff7f0e',
    'purple': '#9467bd'
}

CATEGORY_COLORS = {
    'Regular Menu': '#DA291C',
    'McCafe Menu': '#8B4513',
    'Beverages Menu': '#1E90FF',
    'Breakfast Menu': '#FFC72C',
    'Gourmet Menu': '#9932CC',
    'Condiments Menu': '#228B22',
    'Desserts Menu': '#FF69B4'
}

print("✅ All libraries loaded successfully!")
print("📊 Using Plotly for interactive visualizations")

✅ All libraries loaded successfully!
📊 Using Plotly for interactive visualizations


### 1.2 Load and Preview the Data

In [3]:
# Load the McDonald's nutrition dataset
df = pd.read_csv('./data/macdonald.csv')

print("="*60)
print("🍔 McDONALD'S INDIA MENU - NUTRITION DATASET")
print("="*60)
print(f"\n📊 Dataset: {df.shape[0]} menu items × {df.shape[1]} columns")

# Define column groups
categorical_cols = ['Menu Category', 'Menu Items', 'gOrml']
nutritional_cols = ['Energy (kCal)', 'Protein (g)', 'Total fat (g)', 'Sat Fat (g)',
                    'Trans fat (g)', 'Cholesterols (mg)', 'Total carbohydrate (g)',
                    'Total Sugars (g)', 'Added Sugars (g)', 'Sodium (mg)']

print(f"\n📋 Nutritional variables for PCA: {len(nutritional_cols)}")
for col in nutritional_cols:
    print(f"   • {col}")

# Display sample
df.head()

🍔 McDONALD'S INDIA MENU - NUTRITION DATASET

📊 Dataset: 141 menu items × 14 columns

📋 Nutritional variables for PCA: 10
   • Energy (kCal)
   • Protein (g)
   • Total fat (g)
   • Sat Fat (g)
   • Trans fat (g)
   • Cholesterols (mg)
   • Total carbohydrate (g)
   • Total Sugars (g)
   • Added Sugars (g)
   • Sodium (mg)


,Menu Category,Menu Items,gOrml,Per Serve Size,Energy (kCal),Protein (g),Total fat (g),Sat Fat (g),Trans fat (g),Cholesterols (mg),Total carbohydrate (g),Total Sugars (g),Added Sugars (g),Sodium (mg)
0,Regular Menu,McVeggie™ Burger,g,168.0,402.05,10.24,13.83,5.34,0.16,2.49,56.54,7.90,4.49,706.13
1,Regular Menu,McAloo Tikki Burger®,g,146.0,339.52,8.50,11.31,4.27,0.20,1.47,50.27,7.05,4.07,545.34
2,Regular Menu,McSpicy™ Paneer Burger,g,199.0,652.76,20.29,39.45,17.12,0.18,21.85,52.33,8.35,5.27,1074.58
3,Regular Menu,Spicy Paneer Wrap,g,250.0,674.68,20.96,39.10,19.73,0.26,40.93,59.27,3.50,1.08,1087.46
4,Regular Menu,American Veg Burger,g,177.0,512.17,15.30,23.45,10.51,0.17,25.24,56.96,7.85,4.76,1051.24


### 1.3 Category Distribution

In [4]:
# Visualize menu category distribution
cat_counts = df['Menu Category'].value_counts()

fig = px.pie(
    values=cat_counts.values,
    names=cat_counts.index,
    title='Distribution of Menu Categories',
    color=cat_counts.index,
    color_discrete_map=CATEGORY_COLORS,
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(width=700, height=500)
fig.show()

print("\n📊 Category breakdown:")
for cat, count in cat_counts.items():
    print(f"   {cat}: {count} items ({100*count/len(df):.1f}%)")


📊 Category breakdown:
   McCafe Menu: 51 items (36.2%)
   Regular Menu: 36 items (25.5%)
   Beverages Menu: 17 items (12.1%)
   Breakfast Menu: 15 items (10.6%)
   Gourmet Menu: 11 items (7.8%)
   Condiments Menu: 9 items (6.4%)
   Desserts Menu: 2 items (1.4%)


### 1.4 Check for Missing Values and Prepare Data

In [5]:
# Check missing values
missing = df[nutritional_cols].isnull().sum()
print("Missing values per variable:")
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values in nutritional columns!")

# Extract data for PCA
X_raw = df[nutritional_cols].values
item_names = df['Menu Items'].values
categories = df['Menu Category'].values

# Handle any missing values (for later imputation demonstration)
df_clean = df.dropna(subset=nutritional_cols)
X_raw_clean = df_clean[nutritional_cols].values
item_names_clean = df_clean['Menu Items'].values
categories_clean = df_clean['Menu Category'].values

print(f"\n📈 Data matrix shape: {X_raw_clean.shape[0]} observations × {X_raw_clean.shape[1]} variables")

Missing values per variable:
Sodium (mg)    1
dtype: int64

📈 Data matrix shape: 140 observations × 10 variables


---
# Part 2: Data Preparation for PCA
---

Before applying PCA, we need to standardize the data so all variables contribute equally.

### 2.1 Why Standardization is Essential

PCA finds directions of maximum **variance**. Without standardization:
- Variables with larger scales dominate (e.g., Sodium in mg vs Protein in g)
- The analysis becomes meaningless

After standardization:
- Each variable has mean = 0
- Each variable has standard deviation = 1
- All variables contribute equally to the analysis

In [6]:
# Show why standardization matters - compare variable scales
print("BEFORE STANDARDIZATION - Variable Scales:")
print("="*60)
for i, col in enumerate(nutritional_cols):
    print(f"{col:30s}: mean = {X_raw_clean[:, i].mean():8.2f}, std = {X_raw_clean[:, i].std():8.2f}")

# Standardize the data
scaler = StandardScaler()
X = scaler.fit_transform(X_raw_clean)

print("\n" + "="*60)
print("AFTER STANDARDIZATION:")
print("="*60)
print(f"Mean of each variable: {X.mean(axis=0).round(10)}")
print(f"Std of each variable:  {X.std(axis=0).round(2)}")

BEFORE STANDARDIZATION - Variable Scales:
Energy (kCal)                 : mean =   243.22, std =   184.79
Protein (g)                   : mean =     7.36, std =     8.19
Total fat (g)                 : mean =     9.94, std =    10.32
Sat Fat (g)                   : mean =     5.00, std =     4.90
Trans fat (g)                 : mean =     0.69, std =     6.33
Cholesterols (mg)             : mean =    26.08, std =    50.23
Total carbohydrate (g)        : mean =    31.10, std =    20.58
Total Sugars (g)              : mean =    15.51, std =    15.68
Added Sugars (g)              : mean =    10.37, std =    14.28
Sodium (mg)                   : mean =   362.06, std =   471.47

AFTER STANDARDIZATION:
Mean of each variable: [ 0.  0. -0.  0.  0. -0.  0. -0.  0. -0.]
Std of each variable:  [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### 2.2 Covariance Matrix Heatmap

**What this plot shows:**
- A matrix visualization of correlations between all pairs of nutritional variables
- **Diagonal elements**: Always 1 (variable correlated with itself)
- **Off-diagonal elements**: Correlation between different variables (-1 to +1)

**How to read it:**
- **Red colors (positive)**: Variables that increase together
- **Blue colors (negative)**: Variables that move in opposite directions
- **Values near 0 (white)**: Variables that are independent

**Why it matters for PCA:**
- High correlations indicate redundancy that PCA will compress
- Groups of correlated variables will load on the same principal component

In [7]:
# Compute and visualize the covariance matrix
cov_matrix = np.cov(X.T)

# Create short names for display
short_names = ['Energy', 'Protein', 'TotalFat', 'SatFat', 'TransFat', 
               'Cholest', 'Carbs', 'Sugars', 'AddSugar', 'Sodium']

fig = px.imshow(
    cov_matrix,
    x=short_names,
    y=short_names,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Covariance Matrix (Correlation Matrix for Standardized Data)',
    text_auto='.2f'
)
fig.update_layout(width=800, height=700)
fig.show()

print("\n💡 INSIGHT: High correlations indicate redundancy that PCA will compress.")
print("   Notice strong correlations between: Energy-Fats, Sugars-AddedSugars")


💡 INSIGHT: High correlations indicate redundancy that PCA will compress.
   Notice strong correlations between: Energy-Fats, Sugars-AddedSugars


**📊 Plot Interpretation:**

Looking at the covariance matrix heatmap above:

- **Energy cluster**: Energy, Total fat, Sat Fat, and Cholesterol show strong positive correlations (red) — high-calorie items tend to be fatty
- **Sugar cluster**: Total Sugars and Added Sugars are nearly perfectly correlated — most sugars in fast food are added sugars
- **Carbs are distinct**: Carbohydrates show moderate correlation with sugars but less with fats — distinguishes baked goods from fried items
- **Sodium is semi-independent**: Shows moderate correlation with fats but weak correlation with sugars — sodium is added regardless of sweet/savory

This pattern suggests PCA will likely create components separating fatty/savory items from sweet items.

---
# Part 3: Performing PCA and Extracting Results
---

Now let's fit PCA and extract the key matrices for subsequent visualizations.

In [8]:
# Fit PCA using sklearn
pca = PCA()
T = pca.fit_transform(X)  # Scores matrix (n × k)

# Extract key matrices
P = pca.components_.T      # Loadings matrix (p × k) - transposed for convention
eigenvalues_sklearn = pca.explained_variance_
explained_var_ratio = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var_ratio)

print("PCA MODEL SUMMARY")
print("="*60)
print(f"\n📊 Data matrix X: {X.shape[0]} observations × {X.shape[1]} variables")
print(f"📊 Scores matrix T: {T.shape[0]} observations × {T.shape[1]} components")
print(f"📊 Loadings matrix P: {P.shape[0]} variables × {P.shape[1]} components")

print("\n" + "="*60)
print("VARIANCE EXPLAINED BY EACH COMPONENT")
print("="*60)
for i in range(len(eigenvalues_sklearn)):
    print(f"PC{i+1}: {explained_var_ratio[i]*100:5.2f}% (cumulative: {cumulative_var[i]*100:5.2f}%)")

PCA MODEL SUMMARY

📊 Data matrix X: 140 observations × 10 variables
📊 Scores matrix T: 140 observations × 10 components
📊 Loadings matrix P: 10 variables × 10 components

VARIANCE EXPLAINED BY EACH COMPONENT
PC1: 50.46% (cumulative: 50.46%)
PC2: 24.42% (cumulative: 74.88%)
PC3: 10.57% (cumulative: 85.45%)
PC4:  7.40% (cumulative: 92.85%)
PC5:  3.68% (cumulative: 96.53%)
PC6:  1.72% (cumulative: 98.24%)
PC7:  0.82% (cumulative: 99.06%)
PC8:  0.54% (cumulative: 99.60%)
PC9:  0.36% (cumulative: 99.96%)
PC10:  0.04% (cumulative: 100.00%)


---
# Section 2: Model Validation & Selection
---

> *Aligned with PCAplots.tex Section 2*

One of the first decisions in PCA is: **How many principal components should we keep?** The following plots help answer this question.

### 2.1 Scree Plot *(PCAplots.tex: Slide 2.1)*

**Purpose:**
- Visual component selection
- Identify meaningful PCs vs. noise

**Selection Criteria:**
1. **Elbow method**: Select PCs before the "elbow" where slope flattens
2. **Kaiser criterion**: Keep PCs with eigenvalue λ > 1
3. **Broken stick**: Compare to random expectation

**What to look for:**
- Bars show eigenvalue (variance) for each component
- Sharp drop followed by flat tail → clear separation of signal and noise
- Gradual decline → data is complex, more components may be needed

### 2.2 Explained Variance Plot *(PCAplots.tex: Slide 2.2)*

**Formula:**
$$\text{Individual} = \frac{\lambda_k}{\sum_j \lambda_j} \times 100\%$$
$$\text{Cumulative} = \sum_{i=1}^{k} \text{Individual}_i$$

**Common Thresholds:**
- **80%**: Adequate for exploration
- **90%**: Good for most purposes
- **95%**: Rigorous applications

In [9]:
# Create comprehensive scree plot
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Scree Plot (Eigenvalues)', 'Cumulative Variance Explained']
)

pc_nums = list(range(1, len(eigenvalues_sklearn) + 1))

# Left: Eigenvalue bar chart
fig.add_trace(
    go.Bar(x=pc_nums, y=eigenvalues_sklearn, name='Eigenvalue',
           marker_color=COLORS['blue'], opacity=0.7),
    row=1, col=1
)

# Kaiser criterion line (eigenvalue = 1)
fig.add_hline(y=1, line_dash='dash', line_color='red', row=1, col=1,
              annotation_text='Kaiser Criterion (λ=1)')

# Right: Cumulative variance
fig.add_trace(
    go.Scatter(x=pc_nums, y=cumulative_var * 100, mode='lines+markers',
               name='Cumulative %', line=dict(color=COLORS['green'], width=3),
               marker=dict(size=10)),
    row=1, col=2
)

# Threshold lines
for thresh in [80, 90, 95]:
    fig.add_hline(y=thresh, line_dash='dot', line_color='gray', row=1, col=2,
                  annotation_text=f'{thresh}%')

fig.update_layout(height=450, width=1100, showlegend=False,
                  title_text='Component Selection Criteria')
fig.update_xaxes(title_text='Principal Component', row=1, col=1)
fig.update_xaxes(title_text='Principal Component', row=1, col=2)
fig.update_yaxes(title_text='Eigenvalue', row=1, col=1)
fig.update_yaxes(title_text='Cumulative Variance (%)', row=1, col=2)
fig.show()

# Print recommendations
n_kaiser = (eigenvalues_sklearn > 1).sum()
n_90 = np.argmax(cumulative_var >= 0.90) + 1
n_80 = np.argmax(cumulative_var >= 0.80) + 1

print("\n📊 COMPONENT SELECTION RECOMMENDATIONS")
print("="*50)
print(f"   Kaiser criterion (λ > 1):    {n_kaiser} components")
print(f"   80% variance explained:      {n_80} components")
print(f"   90% variance explained:      {n_90} components")


📊 COMPONENT SELECTION RECOMMENDATIONS
   Kaiser criterion (λ > 1):    3 components
   80% variance explained:      3 components
   90% variance explained:      4 components


**📊 Plot Interpretation for McDonald's Data:**

From the Scree Plot and Cumulative Variance chart:

- **Elbow location**: Clear elbow after PC2 or PC3 — the first 3 components capture the main nutritional patterns
- **Kaiser criterion**: Components 1-3 have eigenvalues > 1, suggesting 3 meaningful nutritional dimensions
- **Variance thresholds**: 
  - 80% variance (3 PCs): Captures main caloric, fat, sugar patterns
  - 90% variance (4 PCs): Adds cholesterol dimension

**What the components represent for McDonald's (actual values):**
1. **PC1 (50.5%)**: "Hearty/Filling Factor" — Energy, Total Fat, Protein, Sodium move together (burgers vs. light items)
2. **PC2 (24.4%)**: "Sweet vs. Savory" — Sugars, Added Sugars, Carbs positive (McFlurry vs. McChicken)
3. **PC3 (10.6%)**: "Trans Fat Indicator" — TransFat dominates (+0.93), flags processed items
4. **PC4 (7.4%)**: "Cholesterol Factor" — Cholesterol (+0.88), distinguishes egg-based items

**Practical implication**: With just 3 numbers (PC1, PC2, PC3), we describe 85% of the nutritional variation across 140 menu items and 10 nutrients.

### 4.2 Variance Explained Table

In [10]:
# Create detailed variance table
variance_df = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(len(eigenvalues_sklearn))],
    'Eigenvalue': eigenvalues_sklearn,
    'Variance %': explained_var_ratio * 100,
    'Cumulative %': cumulative_var * 100,
    'Keep (Kaiser)': eigenvalues_sklearn > 1
})

variance_df.round(3)

,PC,Eigenvalue,Variance %,Cumulative %,Keep (Kaiser)
0,PC1,5.082,50.459,50.459,True
1,PC2,2.459,24.418,74.877,True
2,PC3,1.065,10.570,85.448,True
3,PC4,0.746,7.404,92.851,False
4,PC5,0.370,3.677,96.528,False
5,PC6,0.173,1.715,98.243,False
6,PC7,0.083,0.821,99.064,False
7,PC8,0.054,0.536,99.600,False
8,PC9,0.036,0.358,99.958,False
9,PC10,0.004,0.042,100.000,False


---
# Section 5: Variable Analysis
---

> *Aligned with PCAplots.tex Section 5*

**Loadings** tell us how each original variable contributes to each principal component. This is crucial for interpreting what each PC represents.

### 5.1 Loading Matrix

In [11]:
# Create loadings dataframe
loadings_df = pd.DataFrame(
    P[:, :4],
    index=short_names,
    columns=['PC1', 'PC2', 'PC3', 'PC4']
)

print("LOADING MATRIX (first 4 PCs)")
print("="*60)
print(loadings_df.round(3).to_string())

print("\n💡 Interpretation Guide:")
print("   |loading| > 0.4: Strong contribution")
print("   |loading| 0.2-0.4: Moderate contribution")
print("   |loading| < 0.2: Weak contribution")

LOADING MATRIX (first 4 PCs)
            PC1    PC2    PC3    PC4
Energy    0.420  0.185  0.047 -0.120
Protein   0.419 -0.095  0.076  0.130
TotalFat  0.428 -0.022  0.064 -0.168
SatFat    0.374  0.067 -0.177 -0.234
TransFat  0.048 -0.114  0.934  0.140
Cholest   0.246 -0.129 -0.235  0.884
Carbs     0.268  0.466 -0.049 -0.103
Sugars   -0.085  0.600  0.080  0.166
AddSugar -0.110  0.583  0.117  0.214
Sodium    0.413 -0.063  0.081 -0.009

💡 Interpretation Guide:
   |loading| > 0.4: Strong contribution
   |loading| 0.2-0.4: Moderate contribution
   |loading| < 0.2: Weak contribution


### 5.2 Loading Heatmap *(PCAplots.tex: Slide 5.2)*

**Purpose:**
- Overview of all loadings at once
- Compare multiple PCs simultaneously
- Identify variable patterns and clusters

**Color Coding:**
- **Red**: Positive loading
- **Blue**: Negative loading
- **Intensity**: Magnitude of loading
- **White**: Near zero (variable doesn't contribute)

**What to Look For:**
- Variable clusters (similar patterns across PCs)
- PC structure (which variables dominate each PC)
- Sparse patterns (simple structure)

**Interpretation Guide:**
- |loading| > 0.4: Strong contribution
- |loading| 0.2-0.4: Moderate contribution  
- |loading| < 0.2: Weak contribution

In [12]:
# Visualize loadings as heatmap
fig = px.imshow(
    loadings_df.T,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-0.6, zmax=0.6,
    title='PCA Loadings Heatmap',
    labels=dict(x='Variable', y='Component', color='Loading')
)
fig.update_layout(width=900, height=400)
fig.show()

**📊 Plot Interpretation for McDonald's Data:**

From the loading heatmap (actual values from data):

- **PC1 ("Hearty/Filling Factor")** — 50.5% variance:
  - High positive: TotalFat (+0.43), Energy (+0.42), Protein (+0.42), Sodium (+0.41), SatFat (+0.37)
  - **Menu items**: Big Mac, McSpicy, Chicken Maharaja = high PC1; Apple Slices, Coffee, Soft Drinks = low PC1

- **PC2 ("Sweet vs. Savory")** — 24.4% variance:
  - High positive: Sugars (+0.60), AddSugar (+0.58), Carbs (+0.47)
  - Negative: Cholesterol (-0.13), TransFat (-0.11), Protein (-0.10)
  - **Menu items**: McFlurry, Chocolate Shake, Desserts = high PC2; McChicken, Egg items = low PC2

- **PC3 ("Trans Fat Indicator")** — 10.6% variance:
  - Dominated by: TransFat (+0.93)
  - Negative: Cholesterol (-0.24), SatFat (-0.18)
  - **Menu items**: Items with partially hydrogenated oils = high PC3

- **PC4 ("Cholesterol Factor")** — 7.4% variance:
  - Dominated by: Cholesterol (+0.88)
  - Negative: SatFat (-0.23), TotalFat (-0.17), Energy (-0.12)
  - **Menu items**: Egg-based items (Egg McMuffin) = high PC4; Plant-based items = low PC4

### 5.3 Loading Bar Charts by Component *(PCAplots.tex: Slide 5.1)*

**Purpose:**
- Interpret a single PC
- Identify important variables
- Understand what each PC "means"

**How to Read:**
- **Positive bars (tall)**: Variables that increase with the PC
- **Negative bars (tall)**: Variables that decrease with the PC
- **Large magnitude**: Important variable for this PC
- **Near zero**: Not represented in this PC

**Example Reading:**
- If Protein has high positive loading on PC1, items scoring high on PC1 are protein-rich

In [13]:
# Create loading bar charts for first 4 PCs
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
        f'PC2 ({explained_var_ratio[1]*100:.1f}%)',
        f'PC3 ({explained_var_ratio[2]*100:.1f}%)',
        f'PC4 ({explained_var_ratio[3]*100:.1f}%)'
    ]
)

colors_list = [COLORS['blue'], COLORS['orange'], COLORS['green'], COLORS['purple']]

for i in range(4):
    row = i // 2 + 1
    col = i % 2 + 1
    fig.add_trace(
        go.Bar(x=short_names, y=P[:, i], marker_color=colors_list[i], name=f'PC{i+1}'),
        row=row, col=col
    )
    fig.add_hline(y=0, line_dash='dash', line_color='black', row=row, col=col)

fig.update_layout(height=600, width=1000, showlegend=False,
                  title_text='Loading Profiles by Principal Component')
fig.show()

print("\n💡 INTERPRETATION:")
print("")
print("   PC1: Overall caloric density - high Energy, Fat, Cholesterol")
print("        → Separates hearty meals from light items")
print("")
print("   PC2: Sugar content - high Sugars, low Protein/Fat")
print("        → Separates sweet items (desserts, drinks) from savory")


💡 INTERPRETATION:

   PC1: Overall caloric density - high Energy, Fat, Cholesterol
        → Separates hearty meals from light items

   PC2: Sugar content - high Sugars, low Protein/Fat
        → Separates sweet items (desserts, drinks) from savory


**📊 Plot Interpretation for McDonald's Data:**

From the loading bar charts (actual values):

- **PC1 (50.5%) — "The Hearty/Filling Axis"**:
  - TotalFat (+0.43), Energy (+0.42), Protein (+0.42), Sodium (+0.41) all strongly positive
  - **High PC1 items**: Chicken Maharaja Mac, McSpicy Premium Burger, Big Mac, 5pc Chicken Strips
  - **Low PC1 items**: Plain Coffee, Apple Slices, Ice Tea, Diet drinks
  
- **PC2 (24.4%) — "Sweet vs Savory Axis"**:
  - Sugars (+0.60), AddSugar (+0.58), Carbs (+0.47) strongly positive
  - **High PC2 items**: McFlurry Oreo, Chocolate Shake, Hot Fudge Sundae
  - **Low PC2 items**: Grilled items, Egg McMuffin, plain coffee

- **PC3 (10.6%) — "Trans Fat Indicator"**:
  - TransFat dominates (+0.93), nearly independent of other fats
  - **High PC3 items**: 5pc Chicken Strips (trans fat: 75g!), items with old/reused oil
  - Flags items with partially hydrogenated ingredients

- **PC4 (7.4%) — "Cholesterol Factor"**:
  - Cholesterol dominates (+0.88), negative SatFat (-0.23)
  - **High PC4 items**: Egg-based items (Egg McMuffin, Sausage McMuffin with Egg)

### 1.4 Correlation Loading Plot *(PCAplots.tex: Slide 1.4)*

> Part of **Section 1: Model Interpretation**

**Formula:**
$$\tilde{p}_{jk} = p_{jk} \cdot \sqrt{\frac{\lambda_k}{\text{var}(x_j)}}$$

**Why Use Correlation Loadings (even if you have regular loadings):**
- **Regular loadings** ($p_{jk}$) show direction but aren't bounded — hard to compare across scalings
- **Correlation loadings** are bounded in [-1, 1] → immediate effect size interpretation
- **Radius** gives quality of representation: $r^2 = \text{corr}(x_j, t_1)^2 + \text{corr}(x_j, t_2)^2$

**How to Read:**
- **Variables near outer circle (100%)**: Well-modeled by PC1-PC2
- **Variables inside inner circle (50%)**: Poorly modeled — may need more PCs
- **Arrow direction**: Shows which quadrant items high in that variable will fall
- **Angle between arrows**: Small angle = positively correlated; 90° = uncorrelated; 180° = negatively correlated

In [14]:
# Calculate correlation loadings for all components
corr_loadings = P * np.sqrt(eigenvalues_sklearn)
n_pcs = min(6, len(eigenvalues_sklearn))  # Limit to 6 PCs for dropdown

# Pre-compute all PC combinations
from itertools import product
pc_combos = list(product(range(n_pcs), range(n_pcs)))

# Create figure with all combinations as separate trace groups
fig = go.Figure()

# Add unit circle (always visible)
theta = np.linspace(0, 2*np.pi, 100)
fig.add_trace(go.Scatter(
    x=np.cos(theta), y=np.sin(theta),
    mode='lines', line=dict(color='gray', dash='dash'),
    name='Unit Circle', showlegend=False
))
fig.add_trace(go.Scatter(
    x=0.5*np.cos(theta), y=0.5*np.sin(theta),
    mode='lines', line=dict(color='lightgray', dash='dot'),
    name='', showlegend=False
))
n_base_traces = 2  # circles

# Add traces for each PC combination
traces_per_combo = len(short_names)  # one scatter per variable
for pc_x, pc_y in pc_combos:
    visible = (pc_x == 0 and pc_y == 1)  # Show PC1 vs PC2 initially
    for i, name in enumerate(short_names):
        fig.add_trace(go.Scatter(
            x=[0, corr_loadings[i, pc_x]], 
            y=[0, corr_loadings[i, pc_y]],
            mode='lines+markers+text',
            line=dict(color=COLORS['mcd_red'], width=2),
            marker=dict(size=[0, 10], color=COLORS['mcd_red'], symbol=['circle', 'arrow-up']),
            text=['', name],
            textposition='top center',
            showlegend=False,
            visible=visible,
            hovertemplate=f'{name}<br>PC{pc_x+1}: %{{x:.3f}}<br>PC{pc_y+1}: %{{y:.3f}}<extra></extra>'
        ))

# Add axes
fig.add_hline(y=0, line_color='black', line_width=0.5)
fig.add_vline(x=0, line_color='black', line_width=0.5)

# Create dropdown buttons for X-axis
def make_visibility(selected_x, selected_y):
    """Create visibility array for selected PC combination"""
    vis = [True, True]  # circles always visible
    for pc_x, pc_y in pc_combos:
        is_selected = (pc_x == selected_x and pc_y == selected_y)
        vis.extend([is_selected] * traces_per_combo)
    return vis

x_buttons = []
for i in range(n_pcs):
    # When X changes, default Y to the next PC (or 0 if X is last)
    default_y = (i + 1) % n_pcs if i < n_pcs - 1 else 0
    x_buttons.append(dict(
        label=f'PC{i+1}',
        method='update',
        args=[
            {'visible': make_visibility(i, default_y)},
            {'title': f'Correlation Loading Plot: PC{i+1} vs PC{default_y+1}',
             'xaxis.title': f'PC{i+1} ({explained_var_ratio[i]*100:.1f}%)',
             'yaxis.title': f'PC{default_y+1} ({explained_var_ratio[default_y]*100:.1f}%)'}
        ]
    ))

y_buttons = []
for j in range(n_pcs):
    # When Y changes, keep X as PC1 (index 0)
    y_buttons.append(dict(
        label=f'PC{j+1}',
        method='update',
        args=[
            {'visible': make_visibility(0, j)},
            {'title': f'Correlation Loading Plot: PC1 vs PC{j+1}',
             'xaxis.title': f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
             'yaxis.title': f'PC{j+1} ({explained_var_ratio[j]*100:.1f}%)'}
        ]
    ))

fig.update_layout(
    title=dict(
        text='Correlation Loading Plot: PC1 vs PC2',
        y=0.98,
        x=0.5,
        xanchor='center',
        yanchor='top'
    ),
    xaxis_title=f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
    yaxis_title=f'PC2 ({explained_var_ratio[1]*100:.1f}%)',
    width=800, height=850,
    xaxis=dict(range=[-1.2, 1.2], scaleanchor='y'),
    yaxis=dict(range=[-1.2, 1.2]),
    margin=dict(t=120, b=80, l=80, r=80),
    updatemenus=[
        dict(
            buttons=x_buttons,
            direction='down',
            showactive=True,
            x=0.15, xanchor='left',
            y=1.08, yanchor='top',
            active=0,
            bgcolor='white',
            bordercolor='gray',
            font=dict(size=10)
        ),
        dict(
            buttons=y_buttons,
            direction='down',
            showactive=True,
            x=0.52, xanchor='left',
            y=1.08, yanchor='top',
            active=1,
            bgcolor='white',
            bordercolor='gray',
            font=dict(size=10)
        )
    ],
    annotations=[
        dict(text="<b>X-axis:</b>", x=0.02, xref="paper", y=1.06, yref="paper", 
             showarrow=False, font=dict(size=11), xanchor='left'),
        dict(text="<b>Y-axis:</b>", x=0.39, xref="paper", y=1.06, yref="paper", 
             showarrow=False, font=dict(size=11), xanchor='left')
    ]
)

fig.show()

print("\n💡 TIP: Use the dropdown menus to explore different PC combinations!")
print("   • PC1 vs PC2: Main caloric/sugar patterns (75% variance)")
print("   • PC1 vs PC3: Caloric vs Trans Fat dimension")  
print("   • PC3 vs PC4: Trans Fat vs Cholesterol (specialty nutrients)")


💡 TIP: Use the dropdown menus to explore different PC combinations!
   • PC1 vs PC2: Main caloric/sugar patterns (75% variance)
   • PC1 vs PC3: Caloric vs Trans Fat dimension
   • PC3 vs PC4: Trans Fat vs Cholesterol (specialty nutrients)


**📊 Plot Interpretation for McDonald's Data:**

From the correlation loading plot:

**Well-represented variables (near circle edge):**
- **Energy, Total Fat, Sugars, Added Sugars** — these define the PC1-PC2 space
- If you know these 4 nutrients, you can predict an item's position quite well

**Variable clusters reveal McDonald's menu structure:**
- **Right side cluster (Energy, TotalFat, SatFat, Protein, Sodium)**:
  - These nutrients co-occur → fried items, burgers with cheese, premium sandwiches
  - Example: McSpicy has high values for ALL these (they're bundled together)

- **Top cluster (Total Sugars, Added Sugars, Carbs)**:
  - Almost perfectly aligned (r ≈ 0.98) → McDonald's sweet items use added sugars
  - Example: McFlurry, shakes, desserts cluster here

- **Bottom-left (Protein opposite to Sugars)**:
  - Negative correlation → sweet items aren't protein-rich
  - Example: Grilled wraps (high protein) are opposite to McFlurry (high sugar)

**Poorly represented variable:**
- **Trans Fat** may be near center — it behaves independently from other fats
- This suggests Trans Fat needs PC3 (where it has loading +0.93) to be fully explained

---
# Section 1: Model Interpretation (Scores & Biplots)
---

> *Aligned with PCAplots.tex Section 1*

**Scores** show where each observation (menu item) falls in the principal component space. This is where we see patterns and groupings.

### 1.1 Score Plot ($t_1$ vs $t_2$) *(PCAplots.tex: Slide 1.1)*

**Purpose:**
- Visualize sample relationships
- Detect clusters and trends
- Identify outliers

**What to Look For:**
- **Clusters**: Groups of similar samples
- **Outliers**: Points far from the main group
- **Trends**: Gradients or patterns in the data
- **Separation**: Distinct groups indicating different sample types

**How to Read:**
- **Horizontal position (PC1)**: Based on loading plot interpretation
- **Vertical position (PC2)**: Based on loading plot interpretation
- **Distance from origin**: How "extreme" is this sample

**Interactive tip:** Hover over points to see item names!

In [15]:
# Create interactive score plot with PC selection dropdowns
n_pcs_available = min(6, T.shape[1])

# Prepare data for all PC combinations
score_df = pd.DataFrame({
    **{f'PC{i+1}': T[:, i] for i in range(n_pcs_available)},
    'Item': item_names_clean,
    'Category': categories_clean
})

# Create figure with all PC combinations
from itertools import product
pc_combos = list(product(range(n_pcs_available), range(n_pcs_available)))

fig = go.Figure()

# Add traces for each PC combination and category
categories_unique = score_df['Category'].unique()
traces_per_combo = len(categories_unique)

for pc_x, pc_y in pc_combos:
    visible = (pc_x == 0 and pc_y == 1)  # Show PC1 vs PC2 initially
    for cat in categories_unique:
        mask = score_df['Category'] == cat
        fig.add_trace(go.Scatter(
            x=score_df[f'PC{pc_x+1}'][mask],
            y=score_df[f'PC{pc_y+1}'][mask],
            mode='markers',
            name=cat,
            marker=dict(
                size=10,
                opacity=0.7,
                color=CATEGORY_COLORS.get(cat, '#808080')
            ),
            text=score_df['Item'][mask],
            hovertemplate='<b>%{text}</b><br>' +
                          f'PC{pc_x+1}: %{{x:.2f}}<br>' +
                          f'PC{pc_y+1}: %{{y:.2f}}<extra></extra>',
            visible=visible,
            showlegend=(pc_x == 0 and pc_y == 1)  # Only show legend for initial view
        ))

# Add origin lines (always visible)
fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=0.5)
fig.add_vline(x=0, line_dash='dash', line_color='gray', line_width=0.5)

# Create dropdown buttons
def make_visibility_scores(selected_x, selected_y):
    """Create visibility array for selected PC combination"""
    vis = []
    for pc_x, pc_y in pc_combos:
        is_selected = (pc_x == selected_x and pc_y == selected_y)
        vis.extend([is_selected] * traces_per_combo)
    return vis

x_buttons = []
for i in range(n_pcs_available):
    default_y = (i + 1) % n_pcs_available if i < n_pcs_available - 1 else 0
    x_buttons.append(dict(
        label=f'PC{i+1}',
        method='update',
        args=[
            {'visible': make_visibility_scores(i, default_y)},
            {'title': f'PCA Score Plot: PC{i+1} vs PC{default_y+1}',
             'xaxis.title': f'PC{i+1} ({explained_var_ratio[i]*100:.1f}%)',
             'yaxis.title': f'PC{default_y+1} ({explained_var_ratio[default_y]*100:.1f}%)'}
        ]
    ))

y_buttons = []
for j in range(n_pcs_available):
    y_buttons.append(dict(
        label=f'PC{j+1}',
        method='update',
        args=[
            {'visible': make_visibility_scores(0, j)},
            {'title': f'PCA Score Plot: PC1 vs PC{j+1}',
             'xaxis.title': f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
             'yaxis.title': f'PC{j+1} ({explained_var_ratio[j]*100:.1f}%)'}
        ]
    ))

fig.update_layout(
    title=dict(
        text='PCA Score Plot: Menu Items in PC Space (PC1 vs PC2)',
        y=0.98,
        x=0.5,
        xanchor='center',
        yanchor='top'
    ),
    xaxis_title=f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
    yaxis_title=f'PC2 ({explained_var_ratio[1]*100:.1f}%)',
    width=900, height=850,
    margin=dict(t=120, b=80, l=80, r=80),
    updatemenus=[
        dict(
            buttons=x_buttons,
            direction='down',
            showactive=True,
            x=0.15, xanchor='left',
            y=1.08, yanchor='top',
            active=0,
            bgcolor='white',
            bordercolor='gray',
            font=dict(size=10)
        ),
        dict(
            buttons=y_buttons,
            direction='down',
            showactive=True,
            x=0.52, xanchor='left',
            y=1.08, yanchor='top',
            active=1,
            bgcolor='white',
            bordercolor='gray',
            font=dict(size=10)
        )
    ],
    annotations=[
        dict(text="<b>X-axis:</b>", x=0.02, xref="paper", y=1.06, yref="paper", 
             showarrow=False, font=dict(size=11), xanchor='left'),
        dict(text="<b>Y-axis:</b>", x=0.39, xref="paper", y=1.06, yref="paper", 
             showarrow=False, font=dict(size=11), xanchor='left')
    ]
)

fig.show()

print("\n💡 TIP: Use the dropdown menus to explore different PC combinations!")
print("   • PC1 vs PC2: Caloric vs Sweet patterns (75% variance)")
print("   • PC1 vs PC3: Caloric items vs Trans Fat dimension")
print("   • PC3 vs PC4: Trans Fat vs Cholesterol patterns")
print("\n💡 INSIGHT: Hover over points to see item names!")
print("   • Right side (high PC1): High-calorie items")
print("   • Top (high PC2): Sweet items")
print("   • Bottom-right: Hearty, savory meals")


💡 TIP: Use the dropdown menus to explore different PC combinations!
   • PC1 vs PC2: Caloric vs Sweet patterns (75% variance)
   • PC1 vs PC3: Caloric items vs Trans Fat dimension
   • PC3 vs PC4: Trans Fat vs Cholesterol patterns

💡 INSIGHT: Hover over points to see item names!
   • Right side (high PC1): High-calorie items
   • Top (high PC2): Sweet items
   • Bottom-right: Hearty, savory meals


**📊 Plot Interpretation for McDonald's Data:**

From the score plot:

**Quadrant Analysis (based on loadings):**
| Quadrant | PC1 | PC2 | Nutritional Profile | Example Items |
|----------|-----|-----|---------------------|---------------|
| Upper-right | High | High | High-calorie AND sweet | Chocolate Shake, McFlurry |
| Lower-right | High | Low | High-calorie, savory | Big Mac, McSpicy, Chicken Maharaja |
| Upper-left | Low | High | Light but sweet | Soft Serve Cone, Fruit smoothie |
| Lower-left | Low | Low | Light, savory/neutral | Coffee, Apple Slices, Salads |

**Category Clustering Patterns:**
- **Desserts Menu** (pink): Tight cluster in upper region — ALL desserts are high-sugar
- **Regular Menu** (red): Wide spread on right side — diverse caloric content within this category
- **Beverages Menu** (blue): Split pattern:
  - Near origin: Plain coffee, tea, water
  - Upper area: Shakes, frappes (essentially "liquid desserts")
- **McCafe Menu** (brown): Moderate PC1, elevated PC2 — coffee drinks with flavored syrups
- **Condiments Menu** (green): Tight cluster near origin — ketchup, sauces have minimal nutritional impact

**Interesting Outliers to Explore:**
- Items far right: Largest burgers/combos (e.g., Chicken Maharaja Mac meal)
- Items at top: Highest sugar items (e.g., Large Chocolate Shake ~70g sugar)
- Items near origin but isolated: Check if they're healthier options or data issues

### 1.3 Biplot: Scores and Loadings Combined *(PCAplots.tex: Slide 1.3)*

**Purpose:**
- Combined visualization of samples AND variables
- Direct sample-variable interpretation

**Reading Rules:**
- **Samples in arrow direction** → HIGH values for that variable
- **Samples opposite arrow** → LOW values for that variable
- **Arrow length** → Variable importance in this view
- **Arrow angles** → Correlations between variables

**Example Reading:**
Items on the right side of the plot (if Energy arrow points right) have high energy content.

**Why Biplots are Powerful:**
You can immediately see WHY items are positioned where they are — an item in the direction of the "Sugars" arrow is there BECAUSE it has high sugar content.

In [16]:
# Create biplot with PC selection dropdowns
n_pcs_available = min(6, T.shape[1])

# Pre-compute all PC combinations
from itertools import product
pc_combos = list(product(range(n_pcs_available), range(n_pcs_available)))

fig = go.Figure()

# Add traces for each PC combination
categories_unique = sorted([cat for cat in CATEGORY_COLORS.keys()])
traces_per_combo = len(categories_unique)
scale = 4  # Scale factor for loading arrows

for pc_x, pc_y in pc_combos:
    visible = (pc_x == 0 and pc_y == 1)  # Show PC1 vs PC2 initially
    
    # Add scores (observations) for each category
    for cat in categories_unique:
        mask = categories_clean == cat
        fig.add_trace(go.Scatter(
            x=T[mask, pc_x], y=T[mask, pc_y],
            mode='markers',
            name=cat,
            marker=dict(size=8, color=CATEGORY_COLORS[cat], opacity=0.6),
            text=item_names_clean[mask],
            hovertemplate='%{text}<br>PC' + str(pc_x+1) + ': %{x:.2f}<br>PC' + str(pc_y+1) + ': %{y:.2f}<extra></extra>',
            visible=visible,
            showlegend=(pc_x == 0 and pc_y == 1)  # Only show legend for initial view
        ))

# Add axes (always visible)
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.add_vline(x=0, line_dash='dash', line_color='gray')

# Add loading arrows for PC1 vs PC2 (will be updated via annotations)
for i, name in enumerate(short_names):
    fig.add_annotation(
        x=P[i, 0] * scale, y=P[i, 1] * scale,
        ax=0, ay=0,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1.5, arrowwidth=2,
        arrowcolor='black',
        name=f'arrow_{i}'
    )
    fig.add_annotation(
        x=P[i, 0] * scale * 1.1, y=P[i, 1] * scale * 1.1,
        text=name, showarrow=False, font=dict(size=10, color='black'),
        name=f'label_{i}'
    )

# Create dropdown buttons
def make_visibility_biplot(selected_x, selected_y):
    """Create visibility array for selected PC combination"""
    vis = []
    for pc_x, pc_y in pc_combos:
        is_selected = (pc_x == selected_x and pc_y == selected_y)
        vis.extend([is_selected] * traces_per_combo)
    return vis

def make_annotations(pc_x_idx, pc_y_idx):
    """Create loading arrow annotations for selected PCs"""
    annotations = []
    for i, name in enumerate(short_names):
        # Arrow
        annotations.append(dict(
            x=P[i, pc_x_idx] * scale, y=P[i, pc_y_idx] * scale,
            ax=0, ay=0,
            xref='x', yref='y', axref='x', ayref='y',
            showarrow=True, arrowhead=2, arrowsize=1.5, arrowwidth=2,
            arrowcolor='black'
        ))
        # Label
        annotations.append(dict(
            x=P[i, pc_x_idx] * scale * 1.1, y=P[i, pc_y_idx] * scale * 1.1,
            text=name, showarrow=False, font=dict(size=10, color='black')
        ))
    return annotations

x_buttons = []
for i in range(n_pcs_available):
    default_y = (i + 1) % n_pcs_available if i < n_pcs_available - 1 else 0
    base_annotations = make_annotations(i, default_y)
    # Add dropdown label annotations
    base_annotations.extend([
        dict(text="<b>X-axis:</b>", x=0.02, xref="paper", y=1.06, yref="paper", 
             showarrow=False, font=dict(size=11), xanchor='left'),
        dict(text="<b>Y-axis:</b>", x=0.39, xref="paper", y=1.06, yref="paper", 
             showarrow=False, font=dict(size=11), xanchor='left')
    ])
    x_buttons.append(dict(
        label=f'PC{i+1}',
        method='update',
        args=[
            {'visible': make_visibility_biplot(i, default_y)},
            {'title': f'PCA Biplot: PC{i+1} vs PC{default_y+1}',
             'xaxis.title': f'PC{i+1} ({explained_var_ratio[i]*100:.1f}%)',
             'yaxis.title': f'PC{default_y+1} ({explained_var_ratio[default_y]*100:.1f}%)',
             'annotations': base_annotations}
        ]
    ))

y_buttons = []
for j in range(n_pcs_available):
    base_annotations = make_annotations(0, j)
    # Add dropdown label annotations
    base_annotations.extend([
        dict(text="<b>X-axis:</b>", x=0.02, xref="paper", y=1.06, yref="paper", 
             showarrow=False, font=dict(size=11), xanchor='left'),
        dict(text="<b>Y-axis:</b>", x=0.39, xref="paper", y=1.06, yref="paper", 
             showarrow=False, font=dict(size=11), xanchor='left')
    ])
    y_buttons.append(dict(
        label=f'PC{j+1}',
        method='update',
        args=[
            {'visible': make_visibility_biplot(0, j)},
            {'title': f'PCA Biplot: PC1 vs PC{j+1}',
             'xaxis.title': f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
             'yaxis.title': f'PC{j+1} ({explained_var_ratio[j]*100:.1f}%)',
             'annotations': base_annotations}
        ]
    ))

fig.update_layout(
    title=dict(
        text='PCA Biplot: Scores (Points) + Loadings (Arrows) - PC1 vs PC2',
        y=0.98,
        x=0.5,
        xanchor='center',
        yanchor='top'
    ),
    xaxis_title=f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
    yaxis_title=f'PC2 ({explained_var_ratio[1]*100:.1f}%)',
    width=1000, height=900,
    margin=dict(t=120, b=80, l=80, r=120),
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=1.02),
    updatemenus=[
        dict(
            buttons=x_buttons,
            direction='down',
            showactive=True,
            x=0.15, xanchor='left',
            y=1.08, yanchor='top',
            active=0,
            bgcolor='white',
            bordercolor='gray',
            font=dict(size=10)
        ),
        dict(
            buttons=y_buttons,
            direction='down',
            showactive=True,
            x=0.52, xanchor='left',
            y=1.08, yanchor='top',
            active=1,
            bgcolor='white',
            bordercolor='gray',
            font=dict(size=10)
        )
    ]
)

fig.show()

print("\n💡 TIP: Use the dropdown menus to explore different PC combinations!")
print("   • PC1 vs PC2: Main caloric/sweet patterns with all variable relationships")
print("   • PC1 vs PC3: See how Trans Fat arrow differs from other fats")
print("   • PC3 vs PC4: Compare Trans Fat vs Cholesterol dimensions")
print("\n💡 HOW TO READ A BIPLOT:")
print("   • Points in the direction of an arrow have HIGH values for that variable")
print("   • Points opposite to an arrow have LOW values")
print("   • Arrows pointing same direction = positively correlated variables")


💡 TIP: Use the dropdown menus to explore different PC combinations!
   • PC1 vs PC2: Main caloric/sweet patterns with all variable relationships
   • PC1 vs PC3: See how Trans Fat arrow differs from other fats
   • PC3 vs PC4: Compare Trans Fat vs Cholesterol dimensions

💡 HOW TO READ A BIPLOT:
   • Points in the direction of an arrow have HIGH values for that variable
   • Points opposite to an arrow have LOW values
   • Arrows pointing same direction = positively correlated variables


**📊 Plot Interpretation for McDonald's Data:**

From the biplot:

**Reading Specific Menu Items:**
- **Chicken Maharaja Mac** (far right): Positioned toward Energy/Fat arrows → confirms it's the highest-calorie burger
- **McFlurry Oreo** (upper area): Positioned toward Sugars arrow → confirms high sugar content
- **Grilled Chicken Salad** (lower-left): Opposite to both Fat and Sugar arrows → low-calorie, low-sugar option
- **Large Fries** (right, slightly down): Toward Fat/Sodium arrows, away from Protein → fatty, salty, no protein

**Category-Variable Relationships:**
- **Desserts** cluster exactly where Sugars/AddSugar arrows point — validates these arrows' meaning
- **Egg-based breakfast items** align with Cholesterol arrow — eggs contribute cholesterol
- **Wraps/Grilled options** point toward Protein arrow — confirms they're protein-rich

**Actionable Insights for Menu Analysis:**
| Goal | Look for items... | Examples |
|------|-------------------|----------|
| Lower calories | Opposite to Energy arrow | Coffee, Apple Slices |
| More protein | Along Protein arrow direction | Grilled Wrap, Egg McMuffin |
| Less sugar | Opposite to Sugars arrow | Any burger, savory item |
| Healthier swap | Closer to origin, same quadrant | Regular burger instead of premium |

### 1.5 Combined View: Score Plot + Correlation Loading Plot *(Side by Side)*

**Purpose:**
- See both observations AND variable relationships simultaneously
- Understand how variable loadings relate to sample positions
- Interactive PC selection for comprehensive exploration

In [17]:
# Create side-by-side score plot and correlation loading plot with shared PC selection
from plotly.subplots import make_subplots

n_pcs_available = min(6, T.shape[1])
pc_combos = list(product(range(n_pcs_available), range(n_pcs_available)))

# Prepare score data
score_df = pd.DataFrame({
    **{f'PC{i+1}': T[:, i] for i in range(n_pcs_available)},
    'Item': item_names_clean,
    'Category': categories_clean
})

# Calculate correlation loadings for all components
corr_loadings = P * np.sqrt(eigenvalues_sklearn)

# Create subplots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Score Plot', 'Correlation Loading Plot'),
    horizontal_spacing=0.12,
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}]]
)

# Get unique categories
categories_unique = sorted([cat for cat in CATEGORY_COLORS.keys()])
traces_per_combo = len(categories_unique)

# Add traces for each PC combination
for pc_x, pc_y in pc_combos:
    visible = (pc_x == 0 and pc_y == 1)  # Show PC1 vs PC2 initially
    
    # LEFT SUBPLOT: Score plot
    for cat in categories_unique:
        mask = score_df['Category'] == cat
        fig.add_trace(go.Scatter(
            x=score_df[f'PC{pc_x+1}'][mask],
            y=score_df[f'PC{pc_y+1}'][mask],
            mode='markers',
            name=cat,
            marker=dict(size=8, opacity=0.7, color=CATEGORY_COLORS.get(cat, '#808080')),
            text=score_df['Item'][mask],
            hovertemplate='<b>%{text}</b><br>PC' + str(pc_x+1) + ': %{x:.2f}<br>PC' + str(pc_y+1) + ': %{y:.2f}<extra></extra>',
            visible=visible,
            showlegend=(pc_x == 0 and pc_y == 1),
            legendgroup=cat
        ), row=1, col=1)
    
    # RIGHT SUBPLOT: Loading vectors
    for i, name in enumerate(short_names):
        fig.add_trace(go.Scatter(
            x=[0, corr_loadings[i, pc_x]], 
            y=[0, corr_loadings[i, pc_y]],
            mode='lines+markers+text',
            line=dict(color=COLORS['mcd_red'], width=2),
            marker=dict(size=[0, 8], color=COLORS['mcd_red']),
            text=['', name],
            textposition='top center',
            showlegend=False,
            visible=visible,
            hovertemplate=f'{name}<br>PC{pc_x+1}: %{{x:.3f}}<br>PC{pc_y+1}: %{{y:.3f}}<extra></extra>'
        ), row=1, col=2)

# Add circles to correlation loading plot (always visible)
theta = np.linspace(0, 2*np.pi, 100)
fig.add_trace(go.Scatter(
    x=np.cos(theta), y=np.sin(theta),
    mode='lines', line=dict(color='gray', dash='dash', width=2),
    showlegend=False, hoverinfo='skip'
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=0.5*np.cos(theta), y=0.5*np.sin(theta),
    mode='lines', line=dict(color='gray', dash='dot', width=2),
    showlegend=False, hoverinfo='skip'
), row=1, col=2)

# Add axes
fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=0.5, row=1, col=1)
fig.add_vline(x=0, line_dash='dash', line_color='gray', line_width=0.5, row=1, col=1)
fig.add_hline(y=0, line_color='black', line_width=0.5, row=1, col=2)
fig.add_vline(x=0, line_color='black', line_width=0.5, row=1, col=2)

# Create visibility function
n_vars = len(short_names)

def make_visibility_combined(selected_x, selected_y):
    """Create visibility array for selected PC combination"""
    vis = []
    for pc_x, pc_y in pc_combos:
        is_selected = (pc_x == selected_x and pc_y == selected_y)
        # Score traces (one per category)
        vis.extend([is_selected] * traces_per_combo)
        # Loading vectors (one per variable)
        vis.extend([is_selected] * n_vars)
    # Circles always visible (2 traces at the end)
    vis.extend([True, True])
    return vis

# Create dropdown buttons
x_buttons = []
for i in range(n_pcs_available):
    default_y = (i + 1) % n_pcs_available if i < n_pcs_available - 1 else 0
    x_buttons.append(dict(
        label=f'PC{i+1}',
        method='update',
        args=[
            {'visible': make_visibility_combined(i, default_y)},
            {
                'xaxis.title': f'PC{i+1} ({explained_var_ratio[i]*100:.1f}%)',
                'yaxis.title': f'PC{default_y+1} ({explained_var_ratio[default_y]*100:.1f}%)',
                'xaxis2.title': f'PC{i+1} ({explained_var_ratio[i]*100:.1f}%)',
                'yaxis2.title': f'PC{default_y+1} ({explained_var_ratio[default_y]*100:.1f}%)'
            }
        ]
    ))

y_buttons = []
for j in range(n_pcs_available):
    y_buttons.append(dict(
        label=f'PC{j+1}',
        method='update',
        args=[
            {'visible': make_visibility_combined(0, j)},
            {
                'xaxis.title': f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
                'yaxis.title': f'PC{j+1} ({explained_var_ratio[j]*100:.1f}%)',
                'xaxis2.title': f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
                'yaxis2.title': f'PC{j+1} ({explained_var_ratio[j]*100:.1f}%)'
            }
        ]
    ))

fig.update_layout(
    title=dict(
        text='Combined View: Score Plot (left) + Correlation Loading Plot (right)',
        y=0.98,
        x=0.5,
        xanchor='center',
        yanchor='top'
    ),
    width=1600, height=750,
    margin=dict(t=120, b=80, l=80, r=80),
    updatemenus=[
        dict(
            buttons=x_buttons,
            direction='down',
            showactive=True,
            x=0.15, xanchor='left',
            y=1.08, yanchor='top',
            active=0,
            bgcolor='white',
            bordercolor='gray',
            font=dict(size=10)
        ),
        dict(
            buttons=y_buttons,
            direction='down',
            showactive=True,
            x=0.52, xanchor='left',
            y=1.08, yanchor='top',
            active=1,
            bgcolor='white',
            bordercolor='gray',
            font=dict(size=10)
        )
    ],
    annotations=[
        dict(text="<b>X-axis:</b>", x=0.02, xref="paper", y=1.06, yref="paper", 
             showarrow=False, font=dict(size=11), xanchor='left'),
        dict(text="<b>Y-axis:</b>", x=0.39, xref="paper", y=1.06, yref="paper", 
             showarrow=False, font=dict(size=11), xanchor='left')
    ]
)

# Update axes for score plot
fig.update_xaxes(title_text=f'PC1 ({explained_var_ratio[0]*100:.1f}%)', row=1, col=1)
fig.update_yaxes(title_text=f'PC2 ({explained_var_ratio[1]*100:.1f}%)', row=1, col=1)

# Update axes for correlation loading plot
fig.update_xaxes(title_text=f'PC1 ({explained_var_ratio[0]*100:.1f}%)', 
                 range=[-1.2, 1.2], scaleanchor='y2', row=1, col=2)
fig.update_yaxes(title_text=f'PC2 ({explained_var_ratio[1]*100:.1f}%)', 
                 range=[-1.2, 1.2], row=1, col=2)

fig.show()

print("\n💡 TIP: Use the dropdown menus to explore different PC combinations on BOTH plots!")
print("   • LEFT: See where menu items fall in the PC space")
print("   • RIGHT: See which variables drive those positions")
print("   • Example: Select PC1 vs PC3 to see Trans Fat's unique contribution")


💡 TIP: Use the dropdown menus to explore different PC combinations on BOTH plots!
   • LEFT: See where menu items fall in the PC space
   • RIGHT: See which variables drive those positions
   • Example: Select PC1 vs PC3 to see Trans Fat's unique contribution


---
# Section 3: Outlier Detection & Diagnostics
---

> *Aligned with PCAplots.tex Section 3*

PCA provides powerful tools for detecting unusual observations through two complementary statistics.

### 3.1 Hotelling's T² Statistic *(PCAplots.tex: Slide 3.1)*

**Definition:**
$$T^2_i = \sum_{k=1}^{r} \frac{t_{ik}^2}{\lambda_k}$$

**Critical Limit (F-distribution):**
$$T^2_{\text{crit}} = \frac{r(m-1)}{m-r} F_{r,m-r,\alpha}$$

**Interpretation:**
- Measures distance from center **within** the model
- High T²: extreme but consistent with model structure
- Think: "unusual combination of normal patterns"

---

### 3.2 Q Statistic (SPE / DModX) *(PCAplots.tex: Slide 3.2)*

**Definition (SPE):**
$$Q_i = \|\mathbf{e}_i\|^2 = \|\mathbf{x}_i - \hat{\mathbf{x}}_i\|^2$$

**Interpretation:**
- Distance **outside** the PCA model (residual space)
- High Q: not captured by retained PCs
- Think: "new/unmodeled behavior"

---

### The Four Quadrants in T² vs Q Plot:
| Quadrant | T² | Q | Interpretation |
|----------|-----|---|----------------|
| Normal | Low | Low | Typical observations |
| Leverage | High | Low | Extreme but follows patterns |
| New type | Low | High | Unusual patterns, may need new PC |
| Outlier | High | High | Extreme AND doesn't fit — investigate! |

In [18]:
# Calculate T² and Q with k components
k = 3  # Number of components to use

# Hotelling's T²
T2 = np.sum((T[:, :k] ** 2) / eigenvalues_sklearn[:k], axis=1)

# Q-residual (SPE)
X_reconstructed = T[:, :k] @ P[:, :k].T
E = X - X_reconstructed
Q = np.sum(E ** 2, axis=1)

# Control limits (95% confidence)
n, p = X.shape
alpha = 0.05

# T² limit using F-distribution
T2_limit = k * (n - 1) / (n - k) * f_dist.ppf(1 - alpha, k, n - k)

# Q limit using chi-squared approximation
theta1 = np.sum(eigenvalues_sklearn[k:])
theta2 = np.sum(eigenvalues_sklearn[k:] ** 2)
theta3 = np.sum(eigenvalues_sklearn[k:] ** 3)
h0 = 1 - 2 * theta1 * theta3 / (3 * theta2 ** 2)
ca = stats.norm.ppf(1 - alpha)
Q_limit = theta1 * (ca * np.sqrt(2 * theta2 * h0 ** 2) / theta1 + 
                    1 + theta2 * h0 * (h0 - 1) / theta1 ** 2) ** (1 / h0)

print(f"ANOMALY DETECTION with {k} components")
print("="*60)
print(f"T² limit (95%): {T2_limit:.3f}")
print(f"Q limit (95%):  {Q_limit:.3f}")
print(f"\nItems exceeding T² limit: {(T2 > T2_limit).sum()}")
print(f"Items exceeding Q limit:  {(Q > Q_limit).sum()}")

ANOMALY DETECTION with 3 components
T² limit (95%): 8.129
Q limit (95%):  3.955

Items exceeding T² limit: 6
Items exceeding Q limit:  10


### 3.3 Influence Plot (Q vs T²) *(PCAplots.tex: Slide 3.3)*

**Purpose:**
- Identify outliers and unusual samples
- Distinguish between different types of unusual behavior
- Guide investigation priorities

**Four Quadrants:**
- **Normal (blue, lower-left)**: Typical sample, fits model well
- **High leverage (orange, right)**: Extreme within model — check influence
- **New type (purple, top)**: Doesn't fit model — new behavior
- **Outlier (red, upper-right)**: Extreme AND doesn't fit — investigate root cause

**Actions:**
- **Leverage points**: Check if they unduly influence the model
- **New type**: May need additional PC or model update
- **Outliers**: Investigate data quality or genuinely unusual cases

In [19]:
# Classify points by quadrant
colors_tq = []
for t2, q in zip(T2, Q):
    if t2 > T2_limit and q > Q_limit:
        colors_tq.append('red')      # Both high: extreme outlier
    elif t2 > T2_limit:
        colors_tq.append('orange')   # High T²: extreme in model space
    elif q > Q_limit:
        colors_tq.append('purple')   # High Q: doesn't fit model
    else:
        colors_tq.append(COLORS['blue'])  # Normal

# Create T² vs Q plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=T2, y=Q,
    mode='markers+text',
    marker=dict(size=10, color=colors_tq, opacity=0.7,
                line=dict(width=1, color='black')),
    text=item_names_clean,
    textposition='top center',
    textfont=dict(size=8),
    hovertemplate='<b>%{text}</b><br>T²: %{x:.2f}<br>Q: %{y:.2f}<extra></extra>'
))

# Add control limits
fig.add_vline(x=T2_limit, line_dash='dash', line_color='red',
              annotation_text=f'T² limit = {T2_limit:.2f}')
fig.add_hline(y=Q_limit, line_dash='dash', line_color='red',
              annotation_text=f'Q limit = {Q_limit:.2f}')

# Add quadrant labels
fig.add_annotation(x=T2_limit*0.5, y=Q_limit*0.5, text='NORMAL',
                   showarrow=False, font=dict(size=14, color='green'))
fig.add_annotation(x=T2.max()*0.8, y=Q_limit*0.5, text='Extreme in\nModel Space',
                   showarrow=False, font=dict(size=12, color='orange'))
fig.add_annotation(x=T2_limit*0.5, y=Q.max()*0.8, text='New Behavior\n(Outside Model)',
                   showarrow=False, font=dict(size=12, color='purple'))

fig.update_layout(
    title='T² vs Q Diagnostic Plot (Residual Outlier Map)',
    xaxis_title="Hotelling's T² (variation IN model)",
    yaxis_title='Q Residual (variation OUTSIDE model)',
    width=900, height=700
)
fig.show()

# List outliers
print("\n🚨 OUTLIERS DETECTED:")
print("="*60)
outliers_both = [item_names_clean[i] for i in range(len(T2)) if T2[i] > T2_limit and Q[i] > Q_limit]
outliers_t2 = [item_names_clean[i] for i in range(len(T2)) if T2[i] > T2_limit and Q[i] <= Q_limit]
outliers_q = [item_names_clean[i] for i in range(len(T2)) if T2[i] <= T2_limit and Q[i] > Q_limit]

print(f"\nBoth T² and Q high (extreme outliers): {len(outliers_both)}")
for item in outliers_both[:5]: print(f"   • {item}")
print(f"\nHigh T² only (extreme but follows pattern): {len(outliers_t2)}")
for item in outliers_t2[:5]: print(f"   • {item}")
print(f"\nHigh Q only (unusual pattern): {len(outliers_q)}")
for item in outliers_q[:5]: print(f"   • {item}")


🚨 OUTLIERS DETECTED:

Both T² and Q high (extreme outliers): 1
   • McSpicy Premium Chicken Burger

High T² only (extreme but follows pattern): 5
   • Veg Maharaja Mac
   • 5 piece Chicken Strips
   • Chicken Cheese Lava Burger
   • Large Fanta Oragne
   • Large Sprite

High Q only (unusual pattern): 9
   • Mc Egg Masala Burger
   • Mc Egg Burger for Happy Meal
   • Ghee Rice with Mc Spicy Fried Chicken 1 pc
   • 3 piece Chicken Strips
   • Spicy Egg McMuffin


**📊 Plot Interpretation for McDonald's Data:**

From the T² vs Q plot:

**Quadrant Analysis with McDonald's Examples:**

| Quadrant | T² | Q | What it means | Typical McDonald's items |
|----------|-----|---|---------------|-------------------------|
| **Normal** | Low | Low | Standard menu item | Regular burger, Medium Fries, Coffee |
| **High Leverage** | High | Low | Extreme but predictable | Chicken Maharaja Mac (very high everything, but proportional) |
| **New Behavior** | Low | High | Unexpected nutrient ratio | Item with unusual sodium:fat ratio |
| **Outlier** | High | High | Extreme AND unusual | Premium item with unique recipe |

**What High T² Tells You (McDonald's context):**
- Items are "supersized" versions — more of everything but in typical fast food proportions
- Example: If Big Mac has high T², it's because ALL nutrients are elevated (not a problem, just large)

**What High Q Tells You (McDonald's context):**
- Item breaks the "fast food formula" — e.g., high sodium but LOW fat (unusual for McDonald's)
- Example: If a salad has high Q, its protein:fat ratio doesn't follow the burger pattern

**Investigation Priorities:**
1. **High T² + High Q**: Priority investigation — could be data error or genuinely unique item (e.g., McSpicy Premium Chicken Burger)
2. **High Q only**: Check if ingredient list explains the unusual ratio (e.g., egg items have unusual cholesterol:fat ratio)
3. **High T² only**: No action needed — it's just a large/premium item (e.g., Veg Maharaja Mac)

- Check for: Data entry errors, specialty/limited-time items, regional variations

---
# Section 4: Fault Diagnosis (Contribution Plots)
---

> *Aligned with PCAplots.tex Section 4*

### 4.1 T² Contribution Plot *(PCAplots.tex: Slide 4.1)*

**Formula:**
$$\text{cont}_{T^2,j} = \frac{x_j - \bar{x}_j}{s_j} \sum_{k=1}^{r} \frac{t_k p_{jk}}{\lambda_k}$$

**Use Case:**
- Sample has high T²
- Need to find which variables are responsible
- Bars show contribution magnitude and sign

### 4.2 Q (SPE) Contribution Plot *(PCAplots.tex: Slide 4.2)*

**Formula:**
$$\text{cont}_{Q,j} = e_j^2 = (x_j - \hat{x}_j)^2$$

**Interpretation:**
- Always positive (squared residual)
- Shows which variables don't fit the model
- Large bar = that variable behaves unexpectedly

**Common Causes for High Q Contribution:**
- Sensor malfunction
- Measurement drift
- New process behavior
- Missing correlation

In [20]:
# Find the most extreme outlier
most_extreme_idx = np.argmax(T2 + Q/Q_limit*T2_limit)  # Combined score
extreme_item = item_names_clean[most_extreme_idx]

# Calculate T² contribution for this item
T2_contrib = np.zeros(p)
for j in range(k):
    T2_contrib += (T[most_extreme_idx, j] * P[:, j]) ** 2 / eigenvalues_sklearn[j]

# Q contribution
Q_contrib = E[most_extreme_idx] ** 2

# Create contribution plot
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[f'T² Contribution (Total: {T2[most_extreme_idx]:.2f})',
                    f'Q Contribution (Total: {Q[most_extreme_idx]:.2f})']
)

fig.add_trace(
    go.Bar(x=short_names, y=T2_contrib, marker_color='orange', name='T² contrib'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=short_names, y=Q_contrib, marker_color='purple', name='Q contrib'),
    row=1, col=2
)

fig.update_layout(
    height=400, width=1100, showlegend=False,
    title_text=f'Contribution Plot for: {extreme_item}'
)
fig.show()

# Show actual values for this item
print(f"\n📊 Nutritional values for {extreme_item}:")
for i, col in enumerate(short_names):
    raw_val = X_raw_clean[most_extreme_idx, i]
    std_val = X[most_extreme_idx, i]
    print(f"   {col:12s}: {raw_val:8.1f} (standardized: {std_val:+.2f})")


📊 Nutritional values for 5 piece Chicken Strips:
   Energy      :    411.1 (standardized: +0.91)
   Protein     :     25.4 (standardized: +2.20)
   TotalFat    :     28.5 (standardized: +1.80)
   SatFat      :      0.1 (standardized: -0.99)
   TransFat    :     75.3 (standardized: +11.79)
   Cholest     :      6.7 (standardized: -0.39)
   Carbs       :      0.7 (standardized: -1.48)
   Sugars      :      0.7 (standardized: -0.94)
   AddSugar    :      0.0 (standardized: -0.73)
   Sodium      :   1193.0 (standardized: +1.76)


**📊 Plot Interpretation for McDonald's Data:**

For the selected extreme item:

**T² Contribution — "Why is this item extreme?"**
| Variable with tall bar | What it means for this McDonald's item |
|------------------------|----------------------------------------|
| Energy | Very high/low calorie item (premium burger vs. coffee) |
| Total Fat | Item is very fatty (fried item, cheese-heavy) |
| Sugars | Dessert or sweetened beverage |
| Sodium | Heavily seasoned or contains bacon/cheese |

**Q Contribution — "Why doesn't this item fit the pattern?"**
| Variable with tall bar | What it means (unusual for McDonald's) |
|------------------------|----------------------------------------|
| Protein high | Item has MORE protein than expected given its fat content (unusual) |
| Sodium high | Item has MORE sodium than expected given its calories |
| Sugars high | Savory item with unexpected sweetness (BBQ sauce?) |

| Trans Fat high | Trans fat not following the other fats (old oil?) |

**Example Scenario:**
If a burger shows:
- High T² contribution from **Energy, Fat** → It's just a big burger (normal)
- High Q contribution from **Sodium** → It has unusually high sodium for its size (investigate — extra bacon? sauce?)

**Action based on contribution plots:**
- High T² in Energy/Fat: No issue — it's a premium/large item
- High Q in Sodium: Check recipe — is sodium correct? Compare to similar items
- High Q in Sugars for a burger: Check if BBQ sauce or sweet bun is adding sugar

---
# Section 6: Residual Analysis (Imputation Example)
---

> *Aligned with PCAplots.tex Section 6*

PCA can estimate missing values by exploiting correlations between variables. This demonstrates the predicted vs actual concept from residual analysis.

In [21]:
def pca_imputation(X_missing, n_components=3, max_iter=100, tol=1e-6):
    """
    Impute missing values using iterative PCA (EM-PCA algorithm).
    
    Algorithm:
    1. Initialize missing values with column means
    2. Standardize and fit PCA
    3. Reconstruct data
    4. Update missing values with reconstructed values
    5. Repeat until convergence
    """
    X = X_missing.copy()
    missing_mask = np.isnan(X)
    
    if not missing_mask.any():
        return X, []
    
    # Initialize with column means
    col_means = np.nanmean(X, axis=0)
    for j in range(X.shape[1]):
        X[missing_mask[:, j], j] = col_means[j]
    
    convergence = []
    prev_imputed = X[missing_mask].copy()
    
    for iteration in range(max_iter):
        # Standardize
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Fit PCA and reconstruct
        pca_imp = PCA(n_components=n_components)
        scores = pca_imp.fit_transform(X_scaled)
        X_recon_scaled = scores @ pca_imp.components_
        X_recon = scaler.inverse_transform(X_recon_scaled)
        
        # Update missing values only
        X[missing_mask] = X_recon[missing_mask]
        
        # Check convergence
        current_imputed = X[missing_mask]
        change = np.sqrt(np.mean((current_imputed - prev_imputed) ** 2))
        convergence.append(change)
        
        if change < tol:
            print(f"Converged after {iteration + 1} iterations")
            break
        
        prev_imputed = current_imputed.copy()
    
    return X, convergence

# Demonstration: Create artificial missing values
np.random.seed(42)
X_demo = X_raw_clean.copy()

# Introduce 10 random missing values
n_missing = 10
missing_positions = []
for _ in range(n_missing):
    i = np.random.randint(0, X_demo.shape[0])
    j = np.random.randint(0, X_demo.shape[1])
    missing_positions.append((i, j, X_demo[i, j]))
    X_demo[i, j] = np.nan

print(f"Introduced {n_missing} artificial missing values")

# Perform imputation
X_imputed, conv_history = pca_imputation(X_demo, n_components=3)

# Evaluate
print("\nImputation Results:")
print("="*60)
print(f"{'Item':<30} {'Variable':<12} {'Actual':>8} {'Imputed':>8} {'Error':>8}")
print("-"*60)
errors = []
for i, j, actual in missing_positions:
    imputed = X_imputed[i, j]
    error = imputed - actual
    errors.append(error)
    print(f"{item_names_clean[i][:28]:<30} {short_names[j]:<12} {actual:8.1f} {imputed:8.1f} {error:+8.2f}")

rmse = np.sqrt(np.mean(np.array(errors)**2))
print(f"\nRMSE: {rmse:.2f}")

Introduced 10 artificial missing values
Converged after 17 iterations

Imputation Results:
Item                           Variable       Actual  Imputed    Error
------------------------------------------------------------
Small McFlurry - Oreo          SatFat            2.2      2.1    -0.16
Soft serve cone                Sugars           10.7     11.6    +0.87
4 piece Chicken McNuggets      Carbs            10.5     16.9    +6.36
Medium Thums-up                TotalFat          0.0      0.6    +0.64
Strawberry Green Tea (R)       Sugars            1.6      3.3    +1.63
Large Coca-Cola                SatFat            0.0      3.1    +3.11
Regular McFlurry - Oreo        Sugars           25.4     24.8    -0.57
Vedica Natural Mineral Water   Cholest           0.0     11.5   +11.49
Latte (S)                      Protein           7.1      5.9    -1.25
Raw Mango Cooler               Cholest           0.4     -3.6    -4.02

RMSE: 4.52


### 6.2 Predicted vs Actual Plot *(PCAplots.tex: Slide 6.2)*

**Purpose:**
- Assess reconstruction quality
- One plot per variable
- Identify bias or outliers

**Good Fit Indicators:**
- Points on diagonal line
- High R² (close to 1)
- No systematic deviation

**Problems to Watch For:**
- **Scatter**: Poor prediction
- **Curved pattern**: Nonlinearity
- **Offset**: Bias in model

---

**What the Convergence Plot shows (Left):**
- Algorithm iteratively improves estimates until stabilization
- Steep initial drop → quickly finds good estimates
- Flattening curve → convergence achieved

In [22]:
# Visualize imputation convergence and accuracy
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Convergence History', 'Actual vs Imputed']
)

# Convergence
fig.add_trace(
    go.Scatter(y=conv_history, mode='lines+markers',
               marker=dict(size=6), line=dict(color=COLORS['blue'])),
    row=1, col=1
)

# Actual vs Imputed
actuals = [pos[2] for pos in missing_positions]
imputeds = [X_imputed[pos[0], pos[1]] for pos in missing_positions]

fig.add_trace(
    go.Scatter(x=actuals, y=imputeds, mode='markers',
               marker=dict(size=12, color=COLORS['green'])),
    row=1, col=2
)

# Perfect line
min_val, max_val = min(actuals + imputeds), max(actuals + imputeds)
fig.add_trace(
    go.Scatter(x=[min_val, max_val], y=[min_val, max_val],
               mode='lines', line=dict(dash='dash', color='red'), name='Perfect'),
    row=1, col=2
)

fig.update_layout(height=400, width=1000, showlegend=False)
fig.update_xaxes(title_text='Iteration', row=1, col=1)
fig.update_xaxes(title_text='Actual Value', row=1, col=2)
fig.update_yaxes(title_text='Change (RMSE)', row=1, col=1)
fig.update_yaxes(title_text='Imputed Value', row=1, col=2)
fig.show()

**📊 Plot Interpretation for McDonald's Data:**

From the imputation evaluation plots:

**Convergence analysis:**
- McDonald's data converges quickly (~5-10 iterations) because nutrients are strongly correlated
- Fast food items follow predictable formulas: high fat → high calories → moderate sodium

**Which nutrients are easiest to impute:**
| Easy to impute | Reason | Example |
|----------------|--------|----------|
| Energy | Strongly correlated with Fat, Carbs, Protein | If we know Fat+Carbs+Protein, calories are predictable |
| Added Sugars | Nearly identical to Total Sugars (r≈0.98) | All McDonald's sugar is added sugar |
| Sat Fat | Highly correlated with Total Fat | Fast food fat profile is consistent |

**Harder to impute:**
| Harder to impute | Reason |
|------------------|--------|
| Trans Fat | Varies independently based on oil/frying method |
| Sodium | Added for flavor independently of fat/sugar content |

**Practical use case:**
- If a new menu item is missing Cholesterol data, PCA can estimate it from Energy/Fat values
- Works because McDonald's recipes follow consistent patterns

---
# Part 9: Clustering in PC Space
---

PCA reduces dimensionality, making clustering more effective and interpretable. The following plots help determine the optimal number of clusters and visualize results.

In [23]:
# Cluster using PC scores
scores_for_clustering = T[:, :3]  # First 3 PCs

# Find optimal k using silhouette
k_range = range(2, 8)
silhouette_scores = []

for k_clust in k_range:
    kmeans = KMeans(n_clusters=k_clust, random_state=42, n_init=10)
    labels = kmeans.fit_predict(scores_for_clustering)
    silhouette_scores.append(silhouette_score(scores_for_clustering, labels))

# Plot silhouette scores
fig = px.line(
    x=list(k_range), y=silhouette_scores,
    markers=True,
    title='Optimal Number of Clusters (Silhouette Analysis)',
    labels={'x': 'Number of Clusters (k)', 'y': 'Silhouette Score'}
)
fig.update_layout(width=700, height=400)
fig.show()

# Use optimal k
optimal_k = k_range[np.argmax(silhouette_scores)]
print(f"\nOptimal k by silhouette: {optimal_k}")


Optimal k by silhouette: 6


### 9.1 Silhouette Analysis for Optimal Cluster Selection

**What this plot shows:**
- X-axis: Number of clusters (k)
- Y-axis: Silhouette score (measure of cluster quality)
- Silhouette score ranges from -1 to +1

**How to read it:**
- **Higher values are better**: Items are well-separated from other clusters and close to their own cluster
- **Score > 0.5**: Strong structure found
- **Score 0.25-0.5**: Reasonable structure
- **Score < 0.25**: Weak or artificial structure
- **Peak**: Optimal number of clusters

In [24]:
# Perform clustering with optimal k
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(scores_for_clustering)

# Visualize clusters in PC space
cluster_df = score_df.copy()
cluster_df['Cluster'] = cluster_labels.astype(str)

fig = px.scatter(
    cluster_df, x='PC1', y='PC2',
    color='Cluster',
    hover_name='Item',
    symbol='Category',
    title=f'K-Means Clustering (k={optimal_k}) in PC Space',
    labels={'PC1': f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
            'PC2': f'PC2 ({explained_var_ratio[1]*100:.1f}%)'}
)
fig.update_traces(marker=dict(size=12, opacity=0.7))
fig.update_layout(width=1000, height=700)
fig.show()

# Cluster composition
print("\n📊 CLUSTER COMPOSITION")
print("="*60)
for c in range(optimal_k):
    mask = cluster_labels == c
    print(f"\nCluster {c} ({mask.sum()} items):")
    cluster_cats = pd.Series(categories_clean[mask]).value_counts()
    for cat, count in cluster_cats.items():
        print(f"   {cat}: {count}")


📊 CLUSTER COMPOSITION

Cluster 0 (30 items):
   McCafe Menu: 10
   Condiments Menu: 9
   Breakfast Menu: 7
   Regular Menu: 2
   Beverages Menu: 2

Cluster 1 (47 items):
   Regular Menu: 26
   McCafe Menu: 10
   Breakfast Menu: 8
   Gourmet Menu: 3

Cluster 2 (17 items):
   McCafe Menu: 11
   Beverages Menu: 6

Cluster 3 (14 items):
   Regular Menu: 7
   Gourmet Menu: 7

Cluster 4 (1 items):
   Regular Menu: 1

Cluster 5 (31 items):
   McCafe Menu: 20
   Beverages Menu: 9
   Desserts Menu: 2


### 9.2 Cluster Visualization in PC Space

**What this plot shows:**
- Menu items in PC1-PC2 space, colored by cluster assignment
- Shapes indicate original menu categories
- Shows how data-driven clusters relate to predefined categories

**How to read it:**
- **Well-separated clusters**: Clear gaps between color groups — good clustering
- **Overlapping clusters**: K-means may be forcing artificial boundaries
- **Category vs cluster alignment**: If clusters match categories, menu categories are nutritionally distinct
- **Mixed clusters**: If clusters contain multiple categories, items share nutritional profiles across categories

**📊 Plot Interpretation for McDonald's Data:**

From the silhouette and cluster plots:

**What the clusters likely represent (based on PC1-PC3):**

| Cluster | Likely Nutritional Profile | Example Items |
|---------|---------------------------|---------------|
| 0 | High-calorie savory (burgers, fried) | Big Mac, McSpicy, Chicken Maharaja |
| 1 | Sweet/Dessert items | McFlurry, Shakes, Sundaes |
| 2 | Light/Low-calorie items | Coffee, Apple Slices, Diet drinks |
| 3 | Moderate items | Regular fries, basic burgers, wraps |

**Cluster vs. Menu Category Alignment:**
- **Desserts Menu**: Should align almost perfectly with "Sweet" cluster
- **Regular Menu**: Will SPLIT across multiple clusters (burgers vs. salads vs. wraps)

- **Beverages**: Will SPLIT (shakes with desserts, coffee with light items)
- **Condiments**: All in "Light" cluster (ketchup packets ~15 kCal)

**Key Insight for McDonald's:**
Nutritional clusters don't match marketing categories!
- This reveals that **menu categories are marketing-based, not nutrition-based**
- A chocolate shake (Beverage) clusters with McFlurry (Dessert), not with Coffee (Beverage)

**Business Application:**
- Could relabel menu by nutritional cluster for health-conscious consumers
- Identify "healthier" options within each cluster for recommendations

---
# Section 2 (continued): Cross-Validation for Model Selection
---

> *Aligned with PCAplots.tex Slides 2.3a/2.3b*

Cross-validation helps select the optimal number of components objectively.

In [25]:
# Cross-validation for optimal number of components
def cv_pca(X, n_components, n_folds=5):
    """Calculate cross-validated reconstruction error."""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    errors = []
    
    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        
        # Fit on training data
        scaler_cv = StandardScaler()
        X_train_scaled = scaler_cv.fit_transform(X_train)
        X_test_scaled = scaler_cv.transform(X_test)
        
        pca_cv = PCA(n_components=n_components)
        pca_cv.fit(X_train_scaled)
        
        # Reconstruct test data
        X_test_recon = pca_cv.inverse_transform(pca_cv.transform(X_test_scaled))
        
        # Calculate error
        rmse = np.sqrt(np.mean((X_test_scaled - X_test_recon) ** 2))
        errors.append(rmse)
    
    return np.mean(errors), np.std(errors)

# Test different numbers of components
max_components = min(X_raw_clean.shape) - 1
cv_results = []

print("Cross-validation for component selection...")
for n_comp in range(1, max_components + 1):
    mean_err, std_err = cv_pca(X_raw_clean, n_comp)
    cv_results.append({'n_components': n_comp, 'mean_rmse': mean_err, 'std_rmse': std_err})

cv_df = pd.DataFrame(cv_results)

# Plot CV results
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=cv_df['n_components'],
    y=cv_df['mean_rmse'],
    mode='lines+markers',
    error_y=dict(type='data', array=cv_df['std_rmse']),
    marker=dict(size=10),
    line=dict(width=2)
))

fig.update_layout(
    title='Cross-Validated Reconstruction Error (RMSECV)',
    xaxis_title='Number of Components',
    yaxis_title='Mean CV RMSE',
    width=800, height=500
)
fig.show()

# Find elbow
optimal_cv = cv_df.loc[cv_df['mean_rmse'].diff().abs().idxmax() - 1, 'n_components']
print(f"\n📊 Recommended components by CV elbow: {optimal_cv}")

Cross-validation for component selection...



📊 Recommended components by CV elbow: 2


### 2.3 Cross-Validation Plot *(PCAplots.tex: Slides 2.3a, 2.3b)*

**RMSECV Formula:**
$$\text{RMSECV} = \sqrt{\frac{\sum_{i=1}^{m} \|\mathbf{x}_i - \hat{\mathbf{x}}_{i,-i}\|^2}{m \cdot n}}$$

**Regions on the Plot:**
- **Underfitting region (left)**: Too few components, high error
- **Optimal region (middle)**: Best trade-off
- **Overfitting region (right)**: Components modeling noise

**Rule of Thumb:**
- Choose minimum error point
- Or choose simplest model near the minimum (parsimony)
- Use k-fold CV for more stable curves than leave-one-out

**📊 Plot Interpretation for McDonald's Data:**

From the cross-validation error curve:

**What the curve shows for McDonald's:**
- **Sharp drop (1→3 components)**: First 3 PCs capture real nutritional patterns (calories↔fat, sugars, carbs)
- **Flat region (3-5 components)**: Error stabilizes — we've captured the main "fast food formula"
- **Slight increase (>6 components)**: Overfitting — modeling noise like exact trans fat amounts

**Optimal component selection for McDonald's:**
| Components | What's captured | CV Error | Recommendation |
|------------|----------------|----------|----------------|
| 1 | Just caloric density | High | Underfitting |
| 2 | Calorie + Sugar patterns | Medium | **CV elbow recommendation** |
| 3 | + Trans fat dimension | Lower | **Kaiser criterion (λ>1)** |
| 4-5 | + Cholesterol nuances | Lowest | Best for prediction |
| 6+ | + Noise | Rising | Overfitting |

**Practical recommendation:**
- For **menu analysis/interpretation**: Use 2-3 PCs (CV elbow at 2, Kaiser at 3)
- For **predicting missing nutrients**: Use 3-4 PCs (captures 85-93% variance)
- **Note**: CV elbow (2) and Kaiser (3) give slightly different recommendations — use 3 for interpretability

---
# Summary: Choosing the Right Plot
---

> *Aligned with PCAplots.tex Summary Slides*

In [26]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║            CHOOSING THE RIGHT PLOT - QUICK REFERENCE                         ║
║                    (Aligned with PCAplots.tex)                               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  GOAL                      PRIMARY PLOT              SUPPORTING PLOTS        ║
║  ─────────────────────────────────────────────────────────────────────────   ║
║  Explore sample           Score plot (1.1)          Biplot, 3D scores        ║
║  relationships                                                               ║
║                                                                              ║
║  Understand variables     Loading plot (1.2)        Correlation loadings,    ║
║                                                     heatmap                  ║
║                                                                              ║
║  Select # components      Scree plot (2.1)          Cross-validation,        ║
║                                                     explained variance       ║
║                                                                              ║
║  Find outliers            T² vs Q plot (3.3)        Control charts,          ║
║                                                     influence plot           ║
║                                                                              ║
║  Diagnose faults          Contribution plots        Score contributions      ║
║                           (4.1, 4.2)                                         ║
║                                                                              ║
║  Assess model quality     Residual histogram        Q-Q plot,                ║
║                                                     predicted vs actual      ║
║                                                                              ║
║  Monitor process          Control charts            Batch trajectories       ║
║                                                                              ║
║  Variable selection       VIP / Communalities       Loading bar chart        ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  RECOMMENDED WORKFLOW                                                        ║
║  ─────────────────────────────────────────────────────────────────────────   ║
║  1. VALIDATE:   Scree/CV plots → select components                           ║
║  2. INTERPRET:  Score + Loading plots → understand structure                 ║
║  3. DIAGNOSE:   T²/Q plots → find outliers                                   ║
║  4. VERIFY:     Residual plots → check assumptions                           ║
║  5. MONITOR:    Control charts → ongoing surveillance                        ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  KEY FORMULAS                                                                ║
║  ─────────────────────────────────────────────────────────────────────────   ║
║  PCA Model:          X = TP' + E                                             ║
║  Hotelling's T²:     T²ᵢ = Σₖ (tᵢₖ²/λₖ)                                      ║
║  Q-residual (SPE):   Qᵢ = ‖xᵢ - x̂ᵢ‖²                                        ║
║  VIP:                VIPⱼ = √(n·Σₖ wⱼₖ²·SSₖ / Σₖ SSₖ)                        ║
║  Communality:        hⱼ² = Σₖ pⱼₖ²                                           ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  KEY THRESHOLDS                                                              ║
║  ─────────────────────────────────────────────────────────────────────────   ║
║  Explained variance:  ≥ 80%                                                  ║
║  Kaiser criterion:    λ > 1                                                  ║
║  VIP:                 > 1 = important                                        ║
║  Communality:         > 0.5 = acceptable                                     ║
║  T², Q:               Use F-distribution limits                              ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

print("\n📊 KEY INSIGHTS FOR McDONALD'S DATA:")
print("   • PC1: Caloric density (fatty/heavy vs. light items)")
print("   • PC2: Sweet vs. savory (desserts vs. protein-rich)")
print("   • 3 components capture ~75% variance (Kaiser criterion)")
print("   • Categories cluster by nutritional profile, not just menu placement")


╔══════════════════════════════════════════════════════════════════════════════╗
║            CHOOSING THE RIGHT PLOT - QUICK REFERENCE                         ║
║                    (Aligned with PCAplots.tex)                               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  GOAL                      PRIMARY PLOT              SUPPORTING PLOTS        ║
║  ─────────────────────────────────────────────────────────────────────────   ║
║  Explore sample           Score plot (1.1)          Biplot, 3D scores        ║
║  relationships                                                               ║
║                                                                              ║
║  Understand variables     Loading plot (1.2)        Correlation loadings,    ║
║                                                     heatmap                  ║
║                          

---
# 📊 Interactive Dashboard: Analysis Overview
---

This interactive dashboard provides a comprehensive overview of the entire PCA analysis, allowing you to explore:
- Key PCA metrics and component selection
- Sample and variable relationships
- Outlier detection results
- Model quality diagnostics

Use the dropdowns and sliders to interactively explore different aspects of the analysis.

In [27]:
from ipywidgets import interact, widgets, VBox, HBox, Layout
from IPython.display import display, HTML

# Create interactive dashboard
class PCADashboard:
    def __init__(self, pca_model, scores_df, loadings_df, df_original, X_standardized, 
                 explained_var, eigenvalues, t2_stats, q_stats):
        self.pca = pca_model
        self.scores = scores_df
        self.loadings = loadings_df
        self.df_original = df_original
        self.X_std = X_standardized
        self.explained_var = explained_var
        self.eigenvalues = eigenvalues
        self.t2 = t2_stats
        self.q = q_stats
        
    def create_dashboard(self):
        """Create the complete interactive dashboard"""
        
        # Define widget styles
        style = {'description_width': '150px'}
        layout = Layout(width='400px')
        
        # 1. Overview Section
        overview_widget = widgets.Output()
        with overview_widget:
            self.display_overview()
        
        # 2. Component Selection Widget
        n_comp_slider = widgets.IntSlider(
            value=2, min=2, max=min(5, self.pca.n_components_),
            description='# Components:',
            style=style, layout=layout
        )
        
        # 3. Plot Type Selector
        plot_selector = widgets.Dropdown(
            options=[
                'Score Plot', 
                'Loading Plot', 
                'Biplot',
                'Scree Plot',
                'Explained Variance',
                'Outlier Detection (T² vs Q)',
                'Loading Contributions',
                'T² Contribution',
                'Sample Statistics'
            ],
            value='Score Plot',
            description='Select Plot:',
            style=style,
            layout=layout
        )
        
        # 4. PC Selector for pairwise plots
        pc_x_selector = widgets.Dropdown(
            options=[f'PC{i+1}' for i in range(min(5, self.pca.n_components_))],
            value='PC1',
            description='X-axis:',
            style=style,
            layout=Layout(width='200px')
        )
        
        pc_y_selector = widgets.Dropdown(
            options=[f'PC{i+1}' for i in range(min(5, self.pca.n_components_))],
            value='PC2',
            description='Y-axis:',
            style=style,
            layout=Layout(width='200px')
        )
        
        # 5. Output area for plots
        plot_output = widgets.Output()
        
        # 6. Insights output
        insights_output = widgets.Output()
        
        # Interactive update function
        def update_dashboard(plot_type, pc_x, pc_y, n_comps):
            plot_output.clear_output(wait=True)
            insights_output.clear_output(wait=True)
            
            with plot_output:
                if plot_type == 'Score Plot':
                    self.plot_scores(pc_x, pc_y)
                elif plot_type == 'Loading Plot':
                    self.plot_loadings(pc_x, pc_y)
                elif plot_type == 'Biplot':
                    self.plot_biplot(pc_x, pc_y)
                elif plot_type == 'Scree Plot':
                    self.plot_scree()
                elif plot_type == 'Explained Variance':
                    self.plot_explained_variance(n_comps)
                elif plot_type == 'Outlier Detection (T² vs Q)':
                    self.plot_outliers()
                elif plot_type == 'Loading Contributions':
                    self.plot_loading_contributions(pc_x)
                elif plot_type == 'T² Contribution':
                    self.plot_t2_contributions(n_comps)
                elif plot_type == 'Sample Statistics':
                    self.plot_sample_stats()
            
            with insights_output:
                self.display_insights(plot_type, pc_x, pc_y, n_comps)
        
        # Link widgets to update function
        interactive_plot = widgets.interactive_output(
            update_dashboard,
            {
                'plot_type': plot_selector,
                'pc_x': pc_x_selector,
                'pc_y': pc_y_selector,
                'n_comps': n_comp_slider
            }
        )
        
        # Layout the dashboard
        controls = VBox([
            widgets.HTML('<h3>📊 Dashboard Controls</h3>'),
            plot_selector,
            HBox([pc_x_selector, pc_y_selector]),
            n_comp_slider
        ])
        
        dashboard = VBox([
            widgets.HTML('<h2 style="text-align:center; color:#DA291C;">🍔 PCA Analysis Dashboard - McDonald\'s Nutrition Data</h2>'),
            widgets.HTML('<hr>'),
            overview_widget,
            widgets.HTML('<hr>'),
            controls,
            plot_output,
            insights_output
        ])
        
        display(dashboard)
    
    def display_overview(self):
        """Display key metrics overview"""
        n_samples, n_features = self.X_std.shape
        n_pcs = self.pca.n_components_
        total_var_3pc = np.sum(self.explained_var[:3])
        
        print(f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                          📈 ANALYSIS OVERVIEW                                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Dataset:              McDonald's India Menu Nutrition                       ║
║  Samples:              {n_samples:3d} menu items                                           ║
║  Variables:            {n_features:3d} nutritional features                                  ║
║  Categories:           7 (Regular, McCafe, Beverages, etc.)                  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Principal Components: {n_pcs:3d}                                                         ║
║  Variance (PC1):       {self.explained_var[0]:5.1f}%                                               ║
║  Variance (PC2):       {self.explained_var[1]:5.1f}%                                               ║
║  Variance (PC3):       {self.explained_var[2]:5.1f}%                                               ║
║  Total (3 PCs):        {total_var_3pc:5.1f}%                                               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Outliers (T² > 95%):  {np.sum(self.t2 > np.percentile(self.t2, 95)):3d} samples                                     ║
║  Outliers (Q > 95%):   {np.sum(self.q > np.percentile(self.q, 95)):3d} samples                                     ║
╚══════════════════════════════════════════════════════════════════════════════╝
        """)
    
    def plot_scores(self, pc_x, pc_y):
        """Plot score plot"""
        x_idx = int(pc_x.replace('PC', '')) - 1
        y_idx = int(pc_y.replace('PC', '')) - 1
        
        fig = px.scatter(
            self.scores, x=f'PC{x_idx+1}', y=f'PC{y_idx+1}',
            color='Category',
            color_discrete_map=CATEGORY_COLORS,
            hover_data=['Item'],
            title=f'Score Plot: {pc_x} vs {pc_y}',
            labels={f'PC{x_idx+1}': f'{pc_x} ({self.explained_var[x_idx]:.1f}%)',
                   f'PC{y_idx+1}': f'{pc_y} ({self.explained_var[y_idx]:.1f}%)'}
        )
        fig.update_layout(height=600, width=900)
        fig.show()
    
    def plot_loadings(self, pc_x, pc_y):
        """Plot loading plot"""
        x_idx = int(pc_x.replace('PC', '')) - 1
        y_idx = int(pc_y.replace('PC', '')) - 1
        
        fig = go.Figure()
        
        for i, var in enumerate(self.loadings.columns):
            fig.add_trace(go.Scatter(
                x=[0, self.loadings.iloc[x_idx, i]],
                y=[0, self.loadings.iloc[y_idx, i]],
                mode='lines+markers+text',
                name=var,
                text=['', var],
                textposition='top center',
                line=dict(width=2),
                marker=dict(size=8)
            ))
        
        fig.update_layout(
            title=f'Loading Plot: {pc_x} vs {pc_y}',
            xaxis_title=f'{pc_x} Loadings',
            yaxis_title=f'{pc_y} Loadings',
            height=600, width=900,
            showlegend=True
        )
        fig.show()
    
    def plot_biplot(self, pc_x, pc_y):
        """Plot biplot combining scores and loadings"""
        x_idx = int(pc_x.replace('PC', '')) - 1
        y_idx = int(pc_y.replace('PC', '')) - 1
        
        # Scale loadings for visibility
        scale = 3
        
        fig = go.Figure()
        
        # Add scores
        for cat in self.scores['Category'].unique():
            mask = self.scores['Category'] == cat
            fig.add_trace(go.Scatter(
                x=self.scores.loc[mask, f'PC{x_idx+1}'],
                y=self.scores.loc[mask, f'PC{y_idx+1}'],
                mode='markers',
                name=cat,
                marker=dict(size=8, color=CATEGORY_COLORS.get(cat, '#888888')),
                text=self.scores.loc[mask, 'Item'],
                hovertemplate='%{text}<extra></extra>'
            ))
        
        # Add loadings as arrows
        for i, var in enumerate(self.loadings.columns):
            fig.add_trace(go.Scatter(
                x=[0, self.loadings.iloc[x_idx, i] * scale],
                y=[0, self.loadings.iloc[y_idx, i] * scale],
                mode='lines+text',
                name=var,
                text=['', var],
                textposition='top center',
                line=dict(width=2, color='red'),
                showlegend=False
            ))
        
        fig.update_layout(
            title=f'Biplot: {pc_x} vs {pc_y}',
            xaxis_title=f'{pc_x}',
            yaxis_title=f'{pc_y}',
            height=700, width=1000
        )
        fig.show()
    
    def plot_scree(self):
        """Plot scree plot"""
        fig = go.Figure()
        
        pcs = [f'PC{i+1}' for i in range(len(self.explained_var))]
        
        fig.add_trace(go.Bar(
            x=pcs, y=self.explained_var,
            name='Variance Explained',
            marker_color='#DA291C'
        ))
        
        fig.add_trace(go.Scatter(
            x=pcs, y=np.cumsum(self.explained_var),
            name='Cumulative Variance',
            line=dict(color='#FFC72C', width=3),
            mode='lines+markers'
        ))
        
        fig.add_hline(y=80, line_dash="dash", line_color="green", 
                     annotation_text="80% threshold")
        
        fig.update_layout(
            title='Scree Plot: Variance Explained by Each PC',
            xaxis_title='Principal Component',
            yaxis_title='Variance Explained (%)',
            height=500, width=900
        )
        fig.show()
    
    def plot_explained_variance(self, n_comps):
        """Plot cumulative explained variance"""
        cumvar = np.cumsum(self.explained_var[:n_comps])
        
        fig = go.Figure()
        fig.add_trace(go.Bar(
            x=[f'PC{i+1}' for i in range(n_comps)],
            y=self.explained_var[:n_comps],
            name='Individual',
            marker_color='#DA291C'
        ))
        
        fig.add_trace(go.Scatter(
            x=[f'PC{i+1}' for i in range(n_comps)],
            y=cumvar,
            name='Cumulative',
            mode='lines+markers',
            line=dict(color='#FFC72C', width=3)
        ))
        
        fig.update_layout(
            title=f'Explained Variance (First {n_comps} Components)',
            xaxis_title='Component',
            yaxis_title='Variance (%)',
            height=500, width=800
        )
        fig.show()
        
        print(f"Total variance explained by {n_comps} PCs: {cumvar[-1]:.2f}%")
    
    def plot_outliers(self):
        """Plot T² vs Q for outlier detection"""
        # Calculate thresholds
        n, p = self.X_std.shape
        alpha = 0.05
        t2_limit = (p * (n-1) / (n-p)) * f_dist.ppf(1-alpha, p, n-p)
        q_limit = np.percentile(self.q, 95)
        
        fig = go.Figure()
        
        fig.add_trace(go.Scatter(
            x=self.t2, y=self.q,
            mode='markers',
            marker=dict(
                size=8,
                color=self.scores['Category'].map(CATEGORY_COLORS),
            ),
            text=self.scores['Item'],
            hovertemplate='%{text}<br>T²=%{x:.2f}<br>Q=%{y:.2f}<extra></extra>'
        ))
        
        fig.add_hline(y=q_limit, line_dash="dash", line_color="red",
                     annotation_text="Q limit (95%)")
        fig.add_vline(x=t2_limit, line_dash="dash", line_color="red",
                     annotation_text="T² limit (95%)")
        
        fig.update_layout(
            title='Outlier Detection: Hotelling\'s T² vs Q Residual',
            xaxis_title='Hotelling\'s T²',
            yaxis_title='Q Residual (SPE)',
            height=600, width=900
        )
        fig.show()
        
        outliers_t2 = np.sum(self.t2 > t2_limit)
        outliers_q = np.sum(self.q > q_limit)
        print(f"\n🔍 Outliers detected:")
        print(f"   T² outliers: {outliers_t2}")
        print(f"   Q outliers: {outliers_q}")
    
    def plot_loading_contributions(self, pc):
        """Plot loading contributions for a specific PC"""
        pc_idx = int(pc.replace('PC', '')) - 1
        loadings = self.loadings.iloc[pc_idx]
        
        fig = go.Figure()
        fig.add_trace(go.Bar(
            x=loadings.abs().sort_values(ascending=True).index,
            y=loadings.abs().sort_values(ascending=True).values,
            marker_color='#DA291C',
            orientation='h'
        ))
        
        fig.update_layout(
            title=f'Variable Contributions to {pc} (Absolute Loadings)',
            xaxis_title='Variable',
            yaxis_title='|Loading|',
            height=500, width=800
        )
        fig.show()
    
    def plot_t2_contributions(self, n_comps):
        """Plot T² contributions"""
        # Get top 10 samples by T²
        top_indices = np.argsort(self.t2)[-10:]
        
        contributions = []
        for idx in top_indices:
            t2_contrib = (self.scores.iloc[idx, :n_comps]**2 / 
                         self.eigenvalues[:n_comps]).values
            contributions.append(t2_contrib)
        
        fig = go.Figure()
        
        for i, idx in enumerate(top_indices):
            fig.add_trace(go.Bar(
                name=self.scores.iloc[idx]['Item'][:20],
                x=[f'PC{j+1}' for j in range(n_comps)],
                y=contributions[i]
            ))
        
        fig.update_layout(
            title='T² Contributions - Top 10 Samples',
            xaxis_title='Component',
            yaxis_title='T² Contribution',
            barmode='group',
            height=600, width=1000
        )
        fig.show()
    
    def plot_sample_stats(self):
        """Plot sample statistics summary"""
        stats_df = pd.DataFrame({
            'Item': self.scores['Item'],
            'Category': self.scores['Category'],
            'T²': self.t2,
            'Q': self.q
        })
        
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=('Hotelling\'s T² Distribution', 'Q Residual Distribution')
        )
        
        for cat in stats_df['Category'].unique():
            mask = stats_df['Category'] == cat
            fig.add_trace(
                go.Box(y=stats_df.loc[mask, 'T²'], name=cat, 
                      marker_color=CATEGORY_COLORS.get(cat, '#888888')),
                row=1, col=1
            )
            fig.add_trace(
                go.Box(y=stats_df.loc[mask, 'Q'], name=cat, 
                      marker_color=CATEGORY_COLORS.get(cat, '#888888'),
                      showlegend=False),
                row=1, col=2
            )
        
        fig.update_layout(height=500, width=1200, title_text="Sample Statistics by Category")
        fig.show()
    
    def display_insights(self, plot_type, pc_x, pc_y, n_comps):
        """Display insights based on selected plot"""
        print("\n" + "="*80)
        print(f"💡 INSIGHTS: {plot_type}")
        print("="*80)
        
        if plot_type == 'Score Plot':
            print(f"""
• {pc_x} explains {self.explained_var[int(pc_x[2:])-1]:.1f}% of variance
• {pc_y} explains {self.explained_var[int(pc_y[2:])-1]:.1f}% of variance
• Look for clusters and patterns among categories
• Samples far from origin have distinctive characteristics
            """)
        elif plot_type == 'Loading Plot':
            print("""
• Arrows show how original variables map to principal components
• Long arrows = strong contribution to the PC
• Arrows pointing same direction = positively correlated variables
• Opposite directions = negatively correlated variables
            """)
        elif plot_type == 'Biplot':
            print("""
• Combines score and loading information
• Sample-variable relationships visible
• Samples in direction of arrow have high values for that variable
            """)
        elif plot_type == 'Scree Plot':
            print(f"""
• Kaiser criterion suggests {np.sum(self.eigenvalues > 1)} components (eigenvalue > 1)
• First 3 PCs explain {np.sum(self.explained_var[:3]):.1f}% of total variance
• Look for "elbow" in cumulative variance curve
            """)
        elif plot_type == 'Outlier Detection (T² vs Q)':
            print("""
• T² measures distance from model center (unusual combinations)
• Q measures distance from model plane (not following the pattern)
• High T² & low Q: extreme but following pattern
• Low T² & high Q: moderate values but unusual pattern
            """)
        elif plot_type == 'Loading Contributions':
            print("""
• Shows which variables most strongly influence the selected PC
• Higher values = more important for this component
• Use for variable selection and interpretation
            """)

# Initialize dashboard with computed values
dashboard = PCADashboard(
    pca_model=pca,
    scores_df=score_df,
    loadings_df=loadings_df,
    df_original=df,
    X_standardized=X,
    explained_var=explained_var_ratio * 100,  # Convert to percentage
    eigenvalues=eigenvalues_sklearn,
    t2_stats=T2,
    q_stats=Q
)

# Display the dashboard
dashboard.create_dashboard()

---
# 📸 Export Plots for Presentation
---

Run the cell below to export all key plots as PNG images for the Beamer presentation.

In [28]:
# Install kaleido if not already installed (required for static image export)
try:
    import kaleido
    print("✓ kaleido is already installed")
except ImportError:
    print("Installing kaleido for plot export...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "kaleido", "-q"])
    print("✓ kaleido installed successfully")

import os

# Create output directory
output_dir = "../Presentation"
os.makedirs(output_dir, exist_ok=True)

print("\n" + "="*70)
print("📸 EXPORTING PLOTS FOR BEAMER PRESENTATION")
print("="*70)

# Image export settings
img_width = 1400
img_height = 900

exported = []
failed = []

# 1. Score Plot (PC1 vs PC2)
try:
    fig_score = px.scatter(
        score_df, x='PC1', y='PC2',
        color='Category',
        color_discrete_map=CATEGORY_COLORS,
        hover_data=['Item'],
        title='Score Plot: PC1 vs PC2',
        labels={'PC1': f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
               'PC2': f'PC2 ({explained_var_ratio[1]*100:.1f}%)'}
    )
    fig_score.update_layout(width=img_width, height=img_height, font=dict(size=14))
    fig_score.write_image(f"{output_dir}/score_plot.png")
    exported.append("✓ score_plot.png - Score plot (PC1 vs PC2)")
except Exception as e:
    failed.append(f"✗ score_plot.png: {str(e)}")

# 2. Loading Plot (PC1 vs PC2)
try:
    fig_load = go.Figure()
    for i, var in enumerate(loadings_df.columns):
        fig_load.add_trace(go.Scatter(
            x=[0, loadings_df.iloc[0, i]],
            y=[0, loadings_df.iloc[1, i]],
            mode='lines+markers+text',
            name=var,
            text=['', var],
            textposition='top center',
            line=dict(width=3),
            marker=dict(size=10)
        ))
    fig_load.update_layout(
        title='Loading Plot: PC1 vs PC2',
        xaxis_title='PC1 Loadings',
        yaxis_title='PC2 Loadings',
        width=img_width, height=img_height,
        font=dict(size=14),
        showlegend=True
    )
    fig_load.write_image(f"{output_dir}/loading_plot.png")
    exported.append("✓ loading_plot.png - Loading vectors")
except Exception as e:
    failed.append(f"✗ loading_plot.png: {str(e)}")

# 3. Biplot
try:
    scale = 3
    fig_bi = go.Figure()
    # Add scores
    for cat in score_df['Category'].unique():
        mask = score_df['Category'] == cat
        fig_bi.add_trace(go.Scatter(
            x=score_df.loc[mask, 'PC1'],
            y=score_df.loc[mask, 'PC2'],
            mode='markers',
            name=cat,
            marker=dict(size=8, color=CATEGORY_COLORS.get(cat, '#888888')),
            text=score_df.loc[mask, 'Item'],
            hovertemplate='%{text}<extra></extra>'
        ))
    # Add loadings as arrows
    for i, var in enumerate(loadings_df.columns):
        fig_bi.add_trace(go.Scatter(
            x=[0, loadings_df.iloc[0, i] * scale],
            y=[0, loadings_df.iloc[1, i] * scale],
            mode='lines+text',
            name=var,
            text=['', var],
            textposition='top center',
            line=dict(width=2, color='red'),
            showlegend=False
        ))
    fig_bi.update_layout(
        title='Biplot: Scores and Loadings',
        xaxis_title='PC1',
        yaxis_title='PC2',
        width=img_width, height=img_height,
        font=dict(size=14)
    )
    fig_bi.write_image(f"{output_dir}/biplot.png")
    exported.append("✓ biplot.png - Combined scores and loadings")
except Exception as e:
    failed.append(f"✗ biplot.png: {str(e)}")

# 4. Scree Plot
try:
    pcs_list = [f'PC{i+1}' for i in range(len(explained_var_ratio))]
    fig_scree = go.Figure()
    fig_scree.add_trace(go.Bar(
        x=pcs_list, 
        y=explained_var_ratio * 100,
        name='Variance Explained',
        marker_color='#DA291C'
    ))
    fig_scree.add_trace(go.Scatter(
        x=pcs_list, 
        y=np.cumsum(explained_var_ratio * 100),
        name='Cumulative Variance',
        line=dict(color='#FFC72C', width=3),
        mode='lines+markers'
    ))
    fig_scree.add_hline(y=80, line_dash="dash", line_color="green", 
                       annotation_text="80% threshold")
    fig_scree.update_layout(
        title='Scree Plot: Variance Explained by Each PC',
        xaxis_title='Principal Component',
        yaxis_title='Variance Explained (%)',
        width=img_width, height=img_height,
        font=dict(size=14)
    )
    fig_scree.write_image(f"{output_dir}/scree_plot.png")
    exported.append("✓ scree_plot.png - Variance explained")
except Exception as e:
    failed.append(f"✗ scree_plot.png: {str(e)}")

# 5. T² vs Q Plot
try:
    t2_limit = (p * (n-1) / (n-p)) * f_dist.ppf(0.95, p, n-p)
    q_limit = np.percentile(Q, 95)
    
    fig_tq = go.Figure()
    fig_tq.add_trace(go.Scatter(
        x=T2, y=Q,
        mode='markers',
        marker=dict(
            size=8,
            color=score_df['Category'].map(CATEGORY_COLORS),
        ),
        text=score_df['Item'],
        hovertemplate='%{text}<br>T²=%{x:.2f}<br>Q=%{y:.2f}<extra></extra>'
    ))
    fig_tq.add_hline(y=q_limit, line_dash="dash", line_color="red",
                    annotation_text="Q limit (95%)")
    fig_tq.add_vline(x=t2_limit, line_dash="dash", line_color="red",
                    annotation_text="T² limit (95%)")
    fig_tq.update_layout(
        title='Outlier Detection: Hotelling\'s T² vs Q Residual',
        xaxis_title='Hotelling\'s T²',
        yaxis_title='Q Residual (SPE)',
        width=img_width, height=img_height,
        font=dict(size=14)
    )
    fig_tq.write_image(f"{output_dir}/t2_q_plot.png")
    exported.append("✓ t2_q_plot.png - Outlier detection (T² vs Q)")
except Exception as e:
    failed.append(f"✗ t2_q_plot.png: {str(e)}")

# 6. Covariance/Correlation Heatmap
try:
    fig_cov = px.imshow(
        cov_matrix,
        x=short_names,
        y=short_names,
        color_continuous_scale='RdBu_r',
        zmin=-1, zmax=1,
        title='Correlation Matrix of Standardized Variables',
        text_auto='.2f'
    )
    fig_cov.update_layout(width=img_width, height=img_height, font=dict(size=14))
    fig_cov.write_image(f"{output_dir}/correlation_heatmap.png")
    exported.append("✓ correlation_heatmap.png - Variable correlations")
except Exception as e:
    failed.append(f"✗ correlation_heatmap.png: {str(e)}")

# 7. Loading Bar Chart (first 4 PCs)
try:
    fig_bars = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
            f'PC2 ({explained_var_ratio[1]*100:.1f}%)',
            f'PC3 ({explained_var_ratio[2]*100:.1f}%)',
            f'PC4 ({explained_var_ratio[3]*100:.1f}%)'
        ]
    )
    colors_list = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd']
    for i in range(4):
        row = i // 2 + 1
        col = i % 2 + 1
        fig_bars.add_trace(
            go.Bar(x=short_names, y=P[:, i], marker_color=colors_list[i]),
            row=row, col=col
        )
        fig_bars.add_hline(y=0, line_dash='dash', line_color='black', row=row, col=col)
    fig_bars.update_layout(
        height=img_height, 
        width=img_width, 
        showlegend=False,
        title_text='Loading Profiles by Principal Component',
        font=dict(size=12)
    )
    fig_bars.write_image(f"{output_dir}/loading_bars.png")
    exported.append("✓ loading_bars.png - Loading bar charts")
except Exception as e:
    failed.append(f"✗ loading_bars.png: {str(e)}")

# 8. 3D Score Plot (if clustering was done)
try:
    fig_3d = px.scatter_3d(
        cluster_df, 
        x='PC1', y='PC2', z='PC3',
        color='Cluster',
        hover_data=['Item', 'Category'],
        title='3D Score Plot with Clusters',
        labels={'PC1': f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
               'PC2': f'PC2 ({explained_var_ratio[1]*100:.1f}%)',
               'PC3': f'PC3 ({explained_var_ratio[2]*100:.1f}%)'}
    )
    fig_3d.update_layout(width=img_width, height=img_height, font=dict(size=12))
    fig_3d.write_image(f"{output_dir}/clusters_3d.png")
    exported.append("✓ clusters_3d.png - 3D clustering visualization")
except Exception as e:
    failed.append(f"✗ clusters_3d.png: {str(e)}")

# 9. Q-Q Plot of Residuals
try:
    from scipy import stats
    # Calculate residuals from reconstruction
    residuals_flat = E.flatten()
    
    fig_qq = go.Figure()
    
    # Calculate theoretical quantiles
    sorted_residuals = np.sort(residuals_flat)
    n_points = len(sorted_residuals)
    theoretical_quantiles = stats.norm.ppf(np.linspace(0.01, 0.99, n_points))
    
    # Add scatter plot
    fig_qq.add_trace(go.Scatter(
        x=theoretical_quantiles,
        y=sorted_residuals,
        mode='markers',
        marker=dict(size=4, color='steelblue', opacity=0.6),
        name='Residuals'
    ))
    
    # Add reference line
    min_val = min(theoretical_quantiles.min(), sorted_residuals.min())
    max_val = max(theoretical_quantiles.max(), sorted_residuals.max())
    fig_qq.add_trace(go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode='lines',
        line=dict(color='red', dash='dash', width=2),
        name='Normal Reference'
    ))
    
    fig_qq.update_layout(
        title='Q-Q Plot: Residuals vs Normal Distribution',
        xaxis_title='Theoretical Quantiles',
        yaxis_title='Sample Quantiles (Residuals)',
        width=img_width, height=img_height,
        font=dict(size=14),
        showlegend=True
    )
    fig_qq.write_image(f"{output_dir}/qq_plot.png")
    exported.append("✓ qq_plot.png - Q-Q plot of residuals")
except Exception as e:
    failed.append(f"✗ qq_plot.png: {str(e)}")

# 10. Cross-Validation Error Plot
try:
    fig_cv = go.Figure()
    fig_cv.add_trace(go.Scatter(
        x=cv_df['n_components'],
        y=cv_df['mean_rmse'],
        mode='lines+markers',
        error_y=dict(type='data', array=cv_df['std_rmse']),
        marker=dict(size=10, color='#DA291C'),
        line=dict(width=3, color='#DA291C'),
        name='CV RMSE'
    ))
    fig_cv.add_vline(x=3, line_dash="dash", line_color="green",
                    annotation_text="Optimal (3 PCs)")
    fig_cv.update_layout(
        title='Cross-Validated Reconstruction Error',
        xaxis_title='Number of Components',
        yaxis_title='Mean CV RMSE',
        width=img_width, height=img_height,
        font=dict(size=14)
    )
    fig_cv.write_image(f"{output_dir}/cv_error.png")
    exported.append("✓ cv_error.png - Cross-validation error")
except Exception as e:
    failed.append(f"✗ cv_error.png: {str(e)}")

# 11. Explained Variance (both individual and cumulative)
try:
    fig_var = go.Figure()
    
    # Individual variance
    fig_var.add_trace(go.Bar(
        x=pcs_list[:10],
        y=explained_var_ratio[:10] * 100,
        name='Individual',
        marker_color='#DA291C',
        text=np.round(explained_var_ratio[:10] * 100, 1),
        textposition='outside'
    ))
    
    # Cumulative variance
    fig_var.add_trace(go.Scatter(
        x=pcs_list[:10],
        y=np.cumsum(explained_var_ratio[:10] * 100),
        name='Cumulative',
        mode='lines+markers',
        line=dict(color='#FFC72C', width=4),
        marker=dict(size=12),
        yaxis='y2'
    ))
    
    # Add threshold lines
    fig_var.add_hline(y=80, line_dash="dash", line_color="green",
                     annotation_text="80% threshold", yref='y2')
    
    fig_var.update_layout(
        title='Explained Variance: Individual and Cumulative',
        xaxis_title='Principal Component',
        yaxis=dict(title='Individual Variance (%)', side='left'),
        yaxis2=dict(title='Cumulative Variance (%)', side='right', overlaying='y', range=[0, 100]),
        width=img_width, height=img_height,
        font=dict(size=14),
        barmode='group',
        legend=dict(x=0.7, y=0.5)
    )
    fig_var.write_image(f"{output_dir}/explained_variance.png")
    exported.append("✓ explained_variance.png - Variance explained chart")
except Exception as e:
    failed.append(f"✗ explained_variance.png: {str(e)}")

print("\n" + "="*70)
print("EXPORT SUMMARY")
print("="*70)
print(f"\n✅ Successfully exported {len(exported)} plots:")
for item in exported:
    print(f"   {item}")

if failed:
    print(f"\n❌ Failed to export {len(failed)} plots:")
    for item in failed:
        print(f"   {item}")

print(f"\n📁 All images saved to: {output_dir}/")
print("\n💡 Next steps:")
print("   1. Images are ready in the Presentation folder")
print("   2. Recompile the LaTeX presentation (they're already referenced)")
print("   3. The presentation will now include all plots!")
print("="*70)

✓ kaleido is already installed

📸 EXPORTING PLOTS FOR BEAMER PRESENTATION

EXPORT SUMMARY

✅ Successfully exported 11 plots:
   ✓ score_plot.png - Score plot (PC1 vs PC2)
   ✓ loading_plot.png - Loading vectors
   ✓ biplot.png - Combined scores and loadings
   ✓ scree_plot.png - Variance explained
   ✓ t2_q_plot.png - Outlier detection (T² vs Q)
   ✓ correlation_heatmap.png - Variable correlations
   ✓ loading_bars.png - Loading bar charts
   ✓ clusters_3d.png - 3D clustering visualization
   ✓ qq_plot.png - Q-Q plot of residuals
   ✓ cv_error.png - Cross-validation error
   ✓ explained_variance.png - Variance explained chart

📁 All images saved to: ../Presentation/

💡 Next steps:
   1. Images are ready in the Presentation folder
   2. Recompile the LaTeX presentation (they're already referenced)
   3. The presentation will now include all plots!
